# 04 — Integração INMET + Base Agroambiental

## Projeto AgroESG — Soja | Centro-Oeste e Sul

**Cultura:** soja  
**Regiões:** Centro-Oeste e Sul  
**Período:** 2019–2024

### Objetivo

Integrar os indicadores climáticos do INMET à base agroambiental de soja construída a partir da PAM/IBGE e do MapBiomas Solo.

O processamento climático parte dos arquivos diários da camada Curated do INMET. As estações meteorológicas serão associadas aos municípios por meio de suas coordenadas geográficas e da Malha Municipal do IBGE 2024.

Após a identificação territorial das estações, os dados climáticos serão agregados por município e ano e integrados à base PAM + MapBiomas Solo por `codigo_ibge` e `ano`.

Esta etapa acrescenta a dimensão climática à priorização agroambiental, mas não representa ainda uma estimativa ou certificação de créditos de carbono.

In [2]:
%pip install geopandas

   ---------------------------------------- 0.0/23.8 MB ? eta -:--:--
    --------------------------------------- 0.5/23.8 MB 2.8 MB/s eta 0:00:09
   ---------- ----------------------------- 6.0/23.8 MB 19.2 MB/s eta 0:00:01
   ----------------------------- ---------- 17.8/23.8 MB 35.0 MB/s eta 0:00:01
   ---------------------------------------- 23.8/23.8 MB 37.9 MB/s  0:00:00
   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   ------------- -------------------------- 2.1/6.3 MB 13.0 MB/s eta 0:00:01
   ---------------------------------------- 6.3/6.3 MB 14.4 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 59.4 MB/s  0:00:00

   ---------------------------------------- 0/4 [shapely]
   ---------------------------------------- 0/4 [shapely]
   ---------- ----------------------------- 1/4 [pyproj]
   ---------- ----------------------------- 1/4 [pyproj]
   -------------------- --

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import geopandas as gpd

In [2]:
print("GeoPandas:", gpd.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

GeoPandas: 1.1.4
Pandas: 2.3.3
NumPy: 2.3.5


In [3]:
from pathlib import Path

# Caminho base do projeto
BASE_DIR = Path.cwd().parent

# Caminho da malha municipal de 2024
MALHA_PATH = (
    BASE_DIR
    / "data"
    / "raw"
    / "ibge_territorial"
    / "malha_municipal"
    / "BR_Municipios_2024"
    / "BR_Municipios_2024.shp"
)

print("Caminho da malha:")
print(MALHA_PATH)

print("\nArquivo existe?")
print(MALHA_PATH.exists())

Caminho da malha:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\raw\ibge_territorial\malha_municipal\BR_Municipios_2024\BR_Municipios_2024.shp

Arquivo existe?
True


In [4]:
malha_municipal = gpd.read_file(MALHA_PATH)

print("Dimensão da malha:")
print(malha_municipal.shape)

print("\nCRS:")
print(malha_municipal.crs)

print("\nColunas:")
print(malha_municipal.columns.tolist())

Dimensão da malha:
(5573, 16)

CRS:
EPSG:4674

Colunas:
['CD_MUN', 'NM_MUN', 'CD_RGI', 'NM_RGI', 'CD_RGINT', 'NM_RGINT', 'CD_UF', 'NM_UF', 'SIGLA_UF', 'CD_REGIA', 'NM_REGIA', 'SIGLA_RG', 'CD_CONCU', 'NM_CONCU', 'AREA_KM2', 'geometry']


In [5]:
malha_municipal[
    [
        "CD_MUN",
        "NM_MUN",
        "SIGLA_UF",
        "NM_REGIA",
        "AREA_KM2",
        "geometry"
    ]
].head()

,CD_MUN,NM_MUN,SIGLA_UF,NM_REGIA,AREA_KM2,geometry
0,2504108,Carrapateira,PB,Nordeste,59.070,"POLYGON ((-38.33672 -6.99279, -38.33653 -6.993..."
1,1718451,Pugmil,TO,Norte,401.174,"POLYGON ((-48.91085 -10.53824, -48.911 -10.538..."
2,2104206,Fortuna,MA,Nordeste,835.668,"POLYGON ((-43.95962 -5.49793, -43.96181 -5.497..."
3,5219902,São Francisco de Goiás,GO,Centro-oeste,416.535,"POLYGON ((-49.29477 -16.00852, -49.29484 -16.0..."
4,2708600,São Miguel dos Campos,AL,Nordeste,335.679,"POLYGON ((-36.0739 -9.70094, -36.07339 -9.7008..."


In [6]:
ufs_projeto = [
    "DF", "GO", "MT", "MS",
    "PR", "RS", "SC"
]

malha_recorte = (
    malha_municipal[
        malha_municipal["SIGLA_UF"].isin(ufs_projeto)
    ]
    .copy()
)

print("Dimensão do recorte:")
print(malha_recorte.shape)

print("\nMunicípios por UF:")
print(
    malha_recorte["SIGLA_UF"]
    .value_counts()
    .sort_index()
)

print("\nRegiões presentes:")
print(
    malha_recorte["NM_REGIA"]
    .value_counts()
)

Dimensão do recorte:
(1661, 16)

Municípios por UF:
SIGLA_UF
DF      1
GO    246
MS     79
MT    142
PR    399
RS    499
SC    295
Name: count, dtype: int64

Regiões presentes:
NM_REGIA
Sul             1193
Centro-oeste     468
Name: count, dtype: int64


In [7]:
malha_recorte[
    [
        "CD_MUN",
        "NM_MUN",
        "SIGLA_UF",
        "NM_REGIA",
        "AREA_KM2"
    ]
].sample(10, random_state=42)

,CD_MUN,NM_MUN,SIGLA_UF,NM_REGIA,AREA_KM2
1486,4127908,Tuneiras do Oeste,PR,Sul,698.871
3898,4305132,Cerro Branco,RS,Sul,158.025
2042,5106182,Nova Lacerda,MT,Centro-oeste,4780.426
859,4110300,Inajá,PR,Sul,194.704
3441,5209408,Guarani de Goiás,GO,Centro-oeste,1221.054
1427,4319109,São Martinho,RS,Sul,171.245
2538,5103601,Dom Aquino,MT,Centro-oeste,2183.603
4489,4204103,Caxambu do Sul,SC,Sul,140.873
1609,4124020,Santa Tereza do Oeste,PR,Sul,326.190
2512,4305371,Charrua,RS,Sul,198.748


In [8]:
print("Municípios únicos:")
print(malha_recorte["CD_MUN"].nunique())

print("\nCódigos IBGE duplicados:")
print(malha_recorte["CD_MUN"].duplicated().sum())

print("\nGeometrias ausentes:")
print(malha_recorte["geometry"].isna().sum())

print("\nGeometrias vazias:")
print(malha_recorte.geometry.is_empty.sum())

Municípios únicos:
1661

Códigos IBGE duplicados:
0

Geometrias ausentes:
0

Geometrias vazias:
0


In [10]:
INMET_CURATED_DIR = (
    BASE_DIR
    / "data"
    / "raw"
    / "INMET_LIMPO"
    / "databases_curated"
)

INMET_CENTRO_OESTE_DIR = INMET_CURATED_DIR / "centro_oeste"
INMET_SUL_DIR = INMET_CURATED_DIR / "sul"

print("Centro-Oeste existe?")
print(INMET_CENTRO_OESTE_DIR.exists())

print("\nSul existe?")
print(INMET_SUL_DIR.exists())

Centro-Oeste existe?
True

Sul existe?
True


In [11]:
arquivos_centro_oeste = sorted(
    INMET_CENTRO_OESTE_DIR.glob("*.csv")
)

arquivos_sul = sorted(
    INMET_SUL_DIR.glob("*.csv")
)

print("Arquivos Centro-Oeste:")
print(len(arquivos_centro_oeste))

print("\nArquivos Sul:")
print(len(arquivos_sul))

print("\nTotal de arquivos:")
print(
    len(arquivos_centro_oeste)
    + len(arquivos_sul)
)

Arquivos Centro-Oeste:
2192

Arquivos Sul:
2192

Total de arquivos:
4384


In [12]:
print("Primeiro arquivo Centro-Oeste:")
print(arquivos_centro_oeste[0].name)

print("\nÚltimo arquivo Centro-Oeste:")
print(arquivos_centro_oeste[-1].name)

print("\nPrimeiro arquivo Sul:")
print(arquivos_sul[0].name)

print("\nÚltimo arquivo Sul:")
print(arquivos_sul[-1].name)

Primeiro arquivo Centro-Oeste:
2019-01-01.csv

Último arquivo Centro-Oeste:
2024-12-31.csv

Primeiro arquivo Sul:
2019-01-01.csv

Último arquivo Sul:
2024-12-31.csv


In [13]:
arquivo_exemplo_centro_oeste = arquivos_centro_oeste[0]
arquivo_exemplo_sul = arquivos_sul[0]

df_exemplo_centro_oeste = pd.read_csv(arquivo_exemplo_centro_oeste)
df_exemplo_sul = pd.read_csv(arquivo_exemplo_sul)

print("Arquivo Centro-Oeste:")
print(arquivo_exemplo_centro_oeste.name)

print("\nArquivo Sul:")
print(arquivo_exemplo_sul.name)

Arquivo Centro-Oeste:
2019-01-01.csv

Arquivo Sul:
2019-01-01.csv


In [14]:
print("Dimensão Centro-Oeste:")
print(df_exemplo_centro_oeste.shape)

print("\nDimensão Sul:")
print(df_exemplo_sul.shape)

print("\nColunas Centro-Oeste:")
print(df_exemplo_centro_oeste.columns.tolist())

print("\nColunas Sul:")
print(df_exemplo_sul.columns.tolist())

Dimensão Centro-Oeste:
(111, 38)

Dimensão Sul:
(91, 38)

Colunas Centro-Oeste:
['codigo_wmo', 'estacao', 'uf', 'regiao', 'latitude', 'longitude', 'altitude_m', 'data', 'precipitacao_total_mm', 'pressao_media_mb', 'pressao_max_mb', 'pressao_min_mb', 'radiacao_total_kj_m2', 'temperatura_media_c', 'temperatura_maxima_c', 'temperatura_minima_c', 'ponto_orvalho_medio_c', 'umidade_media_pct', 'umidade_max_pct', 'umidade_min_pct', 'rajada_maxima_ms', 'velocidade_vento_media_ms', 'horas_observadas', 'completude_pct', 'direcao_vento_media_graus', 'choveu', 'chuva_forte', 'classe_chuva', 'temperatura_extrema_alta', 'temperatura_extrema_baixa', 'amplitude_termica_c', 'ventania_extrema', 'vento_forte', 'classe_vento', 'umidade_muito_alta', 'umidade_baixa', 'radiacao_total_mj_m2', 'status_dados']

Colunas Sul:
['codigo_wmo', 'estacao', 'uf', 'regiao', 'latitude', 'longitude', 'altitude_m', 'data', 'precipitacao_total_mm', 'pressao_media_mb', 'pressao_max_mb', 'pressao_min_mb', 'radiacao_total_kj_m

In [15]:
mesmas_colunas = (
    df_exemplo_centro_oeste.columns.tolist()
    == df_exemplo_sul.columns.tolist()
)

print("Os dois arquivos possuem exatamente as mesmas colunas?")
print(mesmas_colunas)

if not mesmas_colunas:
    print("\nColunas apenas no Centro-Oeste:")
    print(
        sorted(
            set(df_exemplo_centro_oeste.columns)
            - set(df_exemplo_sul.columns)
        )
    )

    print("\nColunas apenas no Sul:")
    print(
        sorted(
            set(df_exemplo_sul.columns)
            - set(df_exemplo_centro_oeste.columns)
        )
    )

Os dois arquivos possuem exatamente as mesmas colunas?
True


In [16]:
display(df_exemplo_centro_oeste.head())

display(df_exemplo_sul.head())

,codigo_wmo,estacao,uf,regiao,latitude,longitude,altitude_m,data,precipitacao_total_mm,pressao_media_mb,...,temperatura_extrema_alta,temperatura_extrema_baixa,amplitude_termica_c,ventania_extrema,vento_forte,classe_vento,umidade_muito_alta,umidade_baixa,radiacao_total_mj_m2,status_dados
0,A001,NaN,DF,centro_oeste,-15.789343,-47.925756,1160.96,2019-01-01,0.0,888.025000,...,nao,nao,8.4,nao,nao,moderado,nao,nao,15.5063,dados_completamente_captados
1,A042,NaN,DF,centro_oeste,-15.599722,-48.131111,1143.00,2019-01-01,0.0,889.045833,...,nao,nao,9.2,nao,sim,forte,nao,nao,20.5304,dados_completamente_captados
2,A045,NaN,DF,centro_oeste,-15.596491,-47.625801,1030.36,2019-01-01,0.0,901.237500,...,nao,nao,7.5,nao,nao,moderado,nao,nao,11.2543,dados_completamente_captados
3,A046,NaN,DF,centro_oeste,-15.935278,-48.137500,990.00,2019-01-01,0.0,905.541667,...,nao,nao,8.0,nao,nao,moderado,nao,nao,13.9891,dados_completamente_captados
4,A047,NaN,DF,centro_oeste,-16.012222,-47.557417,1043.00,2019-01-01,0.0,900.054167,...,nao,nao,7.2,nao,nao,moderado,nao,nao,14.3017,dados_completamente_captados


,codigo_wmo,estacao,uf,regiao,latitude,longitude,altitude_m,data,precipitacao_total_mm,pressao_media_mb,...,temperatura_extrema_alta,temperatura_extrema_baixa,amplitude_termica_c,ventania_extrema,vento_forte,classe_vento,umidade_muito_alta,umidade_baixa,radiacao_total_mj_m2,status_dados
0,A807,NaN,PR,sul,-25.448688,-49.230602,922.91,2019-01-01,0.0,912.125000,...,nao,nao,13.6,nao,nao,moderado,nao,nao,27.7361,dados_completamente_captados
1,A818,NaN,PR,sul,-25.010757,-50.853853,803.58,2019-01-01,0.0,923.733333,...,nao,nao,14.2,nao,nao,moderado,nao,nao,30.0698,dados_completamente_captados
2,A819,NaN,PR,sul,-24.786954,-49.999266,993.60,2019-01-01,0.0,904.562500,...,nao,nao,13.8,nao,nao,moderado,nao,nao,28.9097,dados_completamente_captados
3,A820,NaN,PR,sul,-24.533303,-54.019248,392.07,2019-01-01,0.0,965.437500,...,sim,nao,13.8,nao,sim,forte,nao,nao,25.6373,dados_completamente_captados
4,A821,NaN,PR,sul,-23.505266,-49.946387,512.67,2019-01-01,0.0,954.125000,...,nao,nao,12.8,nao,nao,moderado,nao,nao,30.4097,dados_completamente_captados


In [17]:
print("Tipos Centro-Oeste:")
print(df_exemplo_centro_oeste.dtypes)

print("\nTipos Sul:")
print(df_exemplo_sul.dtypes)

Tipos Centro-Oeste:
codigo_wmo                    object
estacao                      float64
uf                            object
regiao                        object
latitude                     float64
longitude                    float64
altitude_m                   float64
data                          object
precipitacao_total_mm        float64
pressao_media_mb             float64
pressao_max_mb               float64
pressao_min_mb               float64
radiacao_total_kj_m2         float64
temperatura_media_c          float64
temperatura_maxima_c         float64
temperatura_minima_c         float64
ponto_orvalho_medio_c        float64
umidade_media_pct            float64
umidade_max_pct              float64
umidade_min_pct              float64
rajada_maxima_ms             float64
velocidade_vento_media_ms    float64
horas_observadas               int64
completude_pct               float64
direcao_vento_media_graus    float64
choveu                        object
chuva_forte       

In [18]:
print("Valores ausentes - Centro-Oeste:")
display(
    df_exemplo_centro_oeste
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

print("\nValores ausentes - Sul:")
display(
    df_exemplo_sul
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

Valores ausentes - Centro-Oeste:


estacao                      111
umidade_min_pct               11
ponto_orvalho_medio_c         11
umidade_max_pct               11
umidade_media_pct             11
direcao_vento_media_graus      9
velocidade_vento_media_ms      8
rajada_maxima_ms               8
temperatura_minima_c           6
amplitude_termica_c            6
pressao_media_mb               6
pressao_max_mb                 6
pressao_min_mb                 6
temperatura_media_c            6
temperatura_maxima_c           6
dtype: int64


Valores ausentes - Sul:


estacao                      91
rajada_maxima_ms              7
velocidade_vento_media_ms     6
umidade_min_pct               5
ponto_orvalho_medio_c         5
umidade_max_pct               5
umidade_media_pct             5
direcao_vento_media_graus     5
amplitude_termica_c           2
pressao_max_mb                2
pressao_min_mb                2
temperatura_media_c           2
temperatura_maxima_c          2
temperatura_minima_c          2
pressao_media_mb              2
dtype: int64

In [19]:
todos_arquivos_inmet = (
    [(arquivo, "centro_oeste") for arquivo in arquivos_centro_oeste]
    +
    [(arquivo, "sul") for arquivo in arquivos_sul]
)

colunas_referencia = df_exemplo_centro_oeste.columns.tolist()

problemas_schema = []
erros_leitura = []

for arquivo, regiao_arquivo in todos_arquivos_inmet:

    try:
        colunas = pd.read_csv(
            arquivo,
            nrows=0
        ).columns.tolist()

        if colunas != colunas_referencia:

            problemas_schema.append(
                {
                    "arquivo": arquivo.name,
                    "regiao": regiao_arquivo,
                    "qtd_colunas": len(colunas),
                    "faltando": sorted(
                        set(colunas_referencia) - set(colunas)
                    ),
                    "extras": sorted(
                        set(colunas) - set(colunas_referencia)
                    )
                }
            )

    except Exception as erro:

        erros_leitura.append(
            {
                "arquivo": arquivo.name,
                "regiao": regiao_arquivo,
                "erro": str(erro)
            }
        )

print("Total de arquivos verificados:")
print(len(todos_arquivos_inmet))

print("\nArquivos com schema diferente:")
print(len(problemas_schema))

print("\nArquivos com erro de leitura:")
print(len(erros_leitura))

Total de arquivos verificados:
4384

Arquivos com schema diferente:
0

Arquivos com erro de leitura:
0


In [20]:
if problemas_schema:
    display(pd.DataFrame(problemas_schema).head(20))
else:
    print("✅ Todos os arquivos possuem o mesmo schema.")

if erros_leitura:
    display(pd.DataFrame(erros_leitura).head(20))
else:
    print("✅ Nenhum erro de leitura encontrado.")

✅ Todos os arquivos possuem o mesmo schema.
✅ Nenhum erro de leitura encontrado.


In [21]:
campos_essenciais = [
    "codigo_wmo",
    "uf",
    "regiao",
    "latitude",
    "longitude",
    "data"
]

print("CENTRO-OESTE")
print(
    df_exemplo_centro_oeste[
        campos_essenciais
    ].isna().sum()
)

print("\nSUL")
print(
    df_exemplo_sul[
        campos_essenciais
    ].isna().sum()
)

CENTRO-OESTE
codigo_wmo    0
uf            0
regiao        0
latitude      0
longitude     0
data          0
dtype: int64

SUL
codigo_wmo    0
uf            0
regiao        0
latitude      0
longitude     0
data          0
dtype: int64


In [22]:
colunas_inmet = [
    "codigo_wmo",
    "uf",
    "regiao",
    "latitude",
    "longitude",
    "altitude_m",
    "data",
    "precipitacao_total_mm",
    "temperatura_media_c",
    "temperatura_maxima_c",
    "temperatura_minima_c",
    "ponto_orvalho_medio_c",
    "umidade_media_pct",
    "umidade_max_pct",
    "umidade_min_pct",
    "radiacao_total_kj_m2",
    "rajada_maxima_ms",
    "velocidade_vento_media_ms",
    "horas_observadas",
    "completude_pct",
    "choveu",
    "chuva_forte",
    "classe_chuva",
    "temperatura_extrema_alta",
    "temperatura_extrema_baixa",
    "amplitude_termica_c",
    "ventania_extrema",
    "vento_forte",
    "classe_vento",
    "umidade_muito_alta",
    "umidade_baixa",
    "radiacao_total_mj_m2",
    "status_dados"
]

print("Quantidade de colunas selecionadas:")
print(len(colunas_inmet))

Quantidade de colunas selecionadas:
33


In [23]:
dfs_centro_oeste = []

for arquivo in arquivos_centro_oeste:
    df_temp = pd.read_csv(
        arquivo,
        usecols=colunas_inmet
    )

    dfs_centro_oeste.append(df_temp)

inmet_centro_oeste = pd.concat(
    dfs_centro_oeste,
    ignore_index=True
)

print("Dimensão Centro-Oeste consolidado:")
print(inmet_centro_oeste.shape)

Dimensão Centro-Oeste consolidado:
(217458, 33)


In [24]:
dfs_sul = []

for arquivo in arquivos_sul:
    df_temp = pd.read_csv(
        arquivo,
        usecols=colunas_inmet
    )

    dfs_sul.append(df_temp)

inmet_sul = pd.concat(
    dfs_sul,
    ignore_index=True
)

print("Dimensão Sul consolidado:")
print(inmet_sul.shape)

Dimensão Sul consolidado:
(203589, 33)


In [25]:
inmet_diario = pd.concat(
    [
        inmet_centro_oeste,
        inmet_sul
    ],
    ignore_index=True
)

print("Dimensão final do INMET diário:")
print(inmet_diario.shape)

print("\nPeríodo:")
print(inmet_diario["data"].min())
print(inmet_diario["data"].max())

print("\nRegiões:")
print(inmet_diario["regiao"].value_counts())

Dimensão final do INMET diário:
(421047, 33)

Período:
2019-01-01
2024-12-31

Regiões:
regiao
centro_oeste    217458
sul             203589
Name: count, dtype: int64


In [26]:
inmet_diario["data"] = pd.to_datetime(
    inmet_diario["data"],
    errors="coerce"
)

inmet_diario["ano"] = inmet_diario["data"].dt.year

print("Tipo da coluna data:")
print(inmet_diario["data"].dtype)

print("\nAnos presentes:")
print(
    inmet_diario["ano"]
    .value_counts()
    .sort_index()
)

Tipo da coluna data:
datetime64[ns]

Anos presentes:
ano
2019    75343
2020    74298
2021    65335
2022    67914
2023    69715
2024    68442
Name: count, dtype: int64


In [27]:
print("Datas inválidas:")
print(inmet_diario["data"].isna().sum())

print("\nRegistros sem código WMO:")
print(inmet_diario["codigo_wmo"].isna().sum())

print("\nRegistros sem latitude:")
print(inmet_diario["latitude"].isna().sum())

print("\nRegistros sem longitude:")
print(inmet_diario["longitude"].isna().sum())

print("\nDuplicidades exatas:")
print(inmet_diario.duplicated().sum())

Datas inválidas:
0

Registros sem código WMO:
0

Registros sem latitude:
0

Registros sem longitude:
0

Duplicidades exatas:
0


In [28]:
print("Quantidade de códigos WMO únicos:")
print(inmet_diario["codigo_wmo"].nunique())

print("\nEstações por UF:")
print(
    inmet_diario[
        ["codigo_wmo", "uf"]
    ]
    .drop_duplicates()
    ["uf"]
    .value_counts()
    .sort_index()
)

Quantidade de códigos WMO únicos:
209

Estações por UF:
uf
DF     5
GO    26
MS    44
MT    39
PR    26
RS    45
SC    24
Name: count, dtype: int64


In [29]:
coordenadas_por_estacao = (
    inmet_diario[
        [
            "codigo_wmo",
            "latitude",
            "longitude"
        ]
    ]
    .drop_duplicates()
)

qtd_coordenadas_por_wmo = (
    coordenadas_por_estacao
    .groupby("codigo_wmo")
    .size()
    .sort_values(ascending=False)
)

print("Maior quantidade de coordenadas para um mesmo WMO:")
print(qtd_coordenadas_por_wmo.max())

print("\nEstações com mais de uma combinação de coordenadas:")
print(
    (qtd_coordenadas_por_wmo > 1).sum()
)

Maior quantidade de coordenadas para um mesmo WMO:
4

Estações com mais de uma combinação de coordenadas:
144


In [30]:
wmos_multiplas_coords = qtd_coordenadas_por_wmo[
    qtd_coordenadas_por_wmo > 1
].index

coords_multiplas = (
    coordenadas_por_estacao[
        coordenadas_por_estacao["codigo_wmo"].isin(wmos_multiplas_coords)
    ]
    .sort_values(
        ["codigo_wmo", "latitude", "longitude"]
    )
)

print("Quantidade de WMO com múltiplas coordenadas:")
print(len(wmos_multiplas_coords))

display(
    coords_multiplas.head(30)
)

Quantidade de WMO com múltiplas coordenadas:
144


,codigo_wmo,latitude,longitude
81220,A001,-15.789444,-47.925833
0,A001,-15.789343,-47.925756
5,A002,-16.642841,-49.220222
148027,A002,-16.642778,-49.220278
6,A003,-17.745066,-49.101698
183076,A003,-17.745000,-49.101667
7,A005,-13.309528,-49.117478
81244,A005,-13.309444,-49.117500
183083,A011,-18.969167,-50.633333
8,A011,-18.969142,-50.633449


In [31]:
variacao_coords = (
    coords_multiplas
    .groupby("codigo_wmo")
    .agg(
        latitude_min=("latitude", "min"),
        latitude_max=("latitude", "max"),
        longitude_min=("longitude", "min"),
        longitude_max=("longitude", "max")
    )
)

variacao_coords["delta_lat"] = (
    variacao_coords["latitude_max"]
    - variacao_coords["latitude_min"]
)

variacao_coords["delta_lon"] = (
    variacao_coords["longitude_max"]
    - variacao_coords["longitude_min"]
)

display(
    variacao_coords
    .sort_values(
        ["delta_lat", "delta_lon"],
        ascending=False
    )
    .head(20)
)

,latitude_min,latitude_max,longitude_min,longitude_max,delta_lat,delta_lon
codigo_wmo,,,,,,
A911,-13.303889,-10.165833,-59.451111,-58.763333,3.138056,0.687778
A895,-27.955278,-27.085311,-52.635711,-52.635556,0.869967,0.000156
A804,-30.842500,-30.750556,-55.613056,-55.401389,0.091944,0.211667
A901,-15.606944,-15.559295,-56.062951,-56.060833,0.047649,0.002118
A912,-15.558889,-15.531389,-55.179444,-55.135556,0.027500,0.043889
A825,-24.183889,-24.158333,-53.048889,-53.030556,0.025556,0.018333
A916,-12.627315,-12.603056,-52.220891,-52.162500,0.024259,0.058391
A704,-20.795000,-20.783333,-51.713333,-51.712222,0.011667,0.001111
A857,-26.786389,-26.776389,-53.514167,-53.504167,0.010000,0.010000


In [32]:
wmos_maior_variacao = (
    variacao_coords
    .assign(
        delta_total=lambda x: (
            x["delta_lat"].abs()
            + x["delta_lon"].abs()
        )
    )
    .sort_values(
        "delta_total",
        ascending=False
    )
    .head(10)
)

display(wmos_maior_variacao)

,latitude_min,latitude_max,longitude_min,longitude_max,delta_lat,delta_lon,delta_total
codigo_wmo,,,,,,,
A911,-13.303889,-10.165833,-59.451111,-58.763333,3.138056,0.687778,3.825833
A895,-27.955278,-27.085311,-52.635711,-52.635556,0.869967,0.000156,0.870122
A804,-30.842500,-30.750556,-55.613056,-55.401389,0.091944,0.211667,0.303611
A916,-12.627315,-12.603056,-52.220891,-52.162500,0.024259,0.058391,0.082650
A912,-15.558889,-15.531389,-55.179444,-55.135556,0.027500,0.043889,0.071389
A819,-24.786954,-24.780000,-50.046389,-49.999167,0.006954,0.047222,0.054176
A901,-15.606944,-15.559295,-56.062951,-56.060833,0.047649,0.002118,0.049767
A825,-24.183889,-24.158333,-53.048889,-53.030556,0.025556,0.018333,0.043889
A857,-26.786389,-26.776389,-53.514167,-53.504167,0.010000,0.010000,0.020000


In [33]:
top_wmos = wmos_maior_variacao.index.tolist()

historico_coords_top = (
    inmet_diario[
        inmet_diario["codigo_wmo"].isin(top_wmos)
    ][
        [
            "codigo_wmo",
            "data",
            "ano",
            "uf",
            "regiao",
            "latitude",
            "longitude"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["codigo_wmo", "data"]
    )
)

display(
    historico_coords_top.head(100)
)

,codigo_wmo,data,ano,uf,regiao,latitude,longitude
33,A704,2019-01-01,2019,MS,centro_oeste,-20.783333,-51.712222
144,A704,2019-01-02,2019,MS,centro_oeste,-20.783333,-51.712222
255,A704,2019-01-03,2019,MS,centro_oeste,-20.783333,-51.712222
366,A704,2019-01-04,2019,MS,centro_oeste,-20.783333,-51.712222
477,A704,2019-01-05,2019,MS,centro_oeste,-20.783333,-51.712222
...,...,...,...,...,...,...,...
10632,A704,2019-04-06,2019,MS,centro_oeste,-20.783333,-51.712222
10745,A704,2019-04-07,2019,MS,centro_oeste,-20.783333,-51.712222
10858,A704,2019-04-08,2019,MS,centro_oeste,-20.783333,-51.712222
10971,A704,2019-04-09,2019,MS,centro_oeste,-20.783333,-51.712222


In [34]:
territorio_por_estacao = (
    inmet_diario[
        [
            "codigo_wmo",
            "uf",
            "regiao"
        ]
    ]
    .drop_duplicates()
)

qtd_territorios_por_wmo = (
    territorio_por_estacao
    .groupby("codigo_wmo")
    .size()
)

print("Estações associadas a mais de uma UF/região:")
print(
    (qtd_territorios_por_wmo > 1).sum()
)

Estações associadas a mais de uma UF/região:
0


In [35]:
wmos_multiplos_territorios = (
    qtd_territorios_por_wmo[
        qtd_territorios_por_wmo > 1
    ]
    .index
)

display(
    territorio_por_estacao[
        territorio_por_estacao["codigo_wmo"]
        .isin(wmos_multiplos_territorios)
    ]
    .sort_values("codigo_wmo")
)

,codigo_wmo,uf,regiao


In [36]:
estacoes_coords = (
    inmet_diario[
        [
            "codigo_wmo",
            "uf",
            "regiao",
            "latitude",
            "longitude"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Combinações únicas WMO + coordenadas:")
print(estacoes_coords.shape)

print("\nCódigos WMO únicos:")
print(estacoes_coords["codigo_wmo"].nunique())

display(estacoes_coords.head())

Combinações únicas WMO + coordenadas:
(360, 5)

Códigos WMO únicos:
209


,codigo_wmo,uf,regiao,latitude,longitude
0,A001,DF,centro_oeste,-15.789343,-47.925756
1,A042,DF,centro_oeste,-15.599722,-48.131111
2,A045,DF,centro_oeste,-15.596491,-47.625801
3,A046,DF,centro_oeste,-15.935278,-48.137500
4,A047,DF,centro_oeste,-16.012222,-47.557417


In [37]:
estacoes_geo = gpd.GeoDataFrame(
    estacoes_coords,
    geometry=gpd.points_from_xy(
        estacoes_coords["longitude"],
        estacoes_coords["latitude"]
    ),
    crs="EPSG:4674"
)

print("CRS das estações:")
print(estacoes_geo.crs)

print("\nQuantidade de pontos:")
print(len(estacoes_geo))

display(estacoes_geo.head())

CRS das estações:
EPSG:4674

Quantidade de pontos:
360


,codigo_wmo,uf,regiao,latitude,longitude,geometry
0,A001,DF,centro_oeste,-15.789343,-47.925756,POINT (-47.92576 -15.78934)
1,A042,DF,centro_oeste,-15.599722,-48.131111,POINT (-48.13111 -15.59972)
2,A045,DF,centro_oeste,-15.596491,-47.625801,POINT (-47.6258 -15.59649)
3,A046,DF,centro_oeste,-15.935278,-48.137500,POINT (-48.1375 -15.93528)
4,A047,DF,centro_oeste,-16.012222,-47.557417,POINT (-47.55742 -16.01222)


In [38]:
malha_join = malha_recorte[
    [
        "CD_MUN",
        "NM_MUN",
        "SIGLA_UF",
        "NM_REGIA",
        "geometry"
    ]
].copy()

estacoes_municipios = gpd.sjoin(
    estacoes_geo,
    malha_join,
    how="left",
    predicate="within"
)

print("Dimensão após spatial join:")
print(estacoes_municipios.shape)

print("\nPontos sem município associado:")
print(estacoes_municipios["CD_MUN"].isna().sum())

print("\nMunicípios com pelo menos uma combinação de estação:")
print(estacoes_municipios["CD_MUN"].nunique())

Dimensão após spatial join:
(360, 11)

Pontos sem município associado:
0

Municípios com pelo menos uma combinação de estação:
205


In [39]:
display(
    estacoes_municipios[
        [
            "codigo_wmo",
            "uf",
            "regiao",
            "latitude",
            "longitude",
            "CD_MUN",
            "NM_MUN",
            "SIGLA_UF",
            "NM_REGIA"
        ]
    ].head(20)
)

,codigo_wmo,uf,regiao,latitude,longitude,CD_MUN,NM_MUN,SIGLA_UF,NM_REGIA
0,A001,DF,centro_oeste,-15.789343,-47.925756,5300108,Brasília,DF,Centro-oeste
1,A042,DF,centro_oeste,-15.599722,-48.131111,5300108,Brasília,DF,Centro-oeste
2,A045,DF,centro_oeste,-15.596491,-47.625801,5300108,Brasília,DF,Centro-oeste
3,A046,DF,centro_oeste,-15.935278,-48.137500,5300108,Brasília,DF,Centro-oeste
4,A047,DF,centro_oeste,-16.012222,-47.557417,5300108,Brasília,DF,Centro-oeste
5,A002,GO,centro_oeste,-16.642841,-49.220222,5208707,Goiânia,GO,Centro-oeste
6,A003,GO,centro_oeste,-17.745066,-49.101698,5213806,Morrinhos,GO,Centro-oeste
7,A005,GO,centro_oeste,-13.309528,-49.117478,5218003,Porangatu,GO,Centro-oeste
8,A011,GO,centro_oeste,-18.969142,-50.633449,5220405,São Simão,GO,Centro-oeste
9,A012,GO,centro_oeste,-16.260556,-47.966944,5212501,Luziânia,GO,Centro-oeste


In [40]:
estacoes_municipios["uf_confere"] = (
    estacoes_municipios["uf"]
    == estacoes_municipios["SIGLA_UF"]
)

print("Pontos com UF divergente:")
print((~estacoes_municipios["uf_confere"]).sum())

Pontos com UF divergente:
1


In [41]:
municipios_por_wmo = (
    estacoes_municipios
    .dropna(subset=["CD_MUN"])
    .groupby("codigo_wmo")["CD_MUN"]
    .nunique()
    .sort_values(ascending=False)
)

print("Maior quantidade de municípios para um mesmo WMO:")
print(municipios_por_wmo.max())

print("\nWMO associados a mais de um município:")
print((municipios_por_wmo > 1).sum())

Maior quantidade de municípios para um mesmo WMO:
2

WMO associados a mais de um município:
3


In [42]:
wmos_multiplos_municipios = municipios_por_wmo[
    municipios_por_wmo > 1
].index

display(
    estacoes_municipios[
        estacoes_municipios["codigo_wmo"].isin(
            wmos_multiplos_municipios
        )
    ][
        [
            "codigo_wmo",
            "uf",
            "latitude",
            "longitude",
            "CD_MUN",
            "NM_MUN",
            "SIGLA_UF"
        ]
    ]
    .sort_values(
        ["codigo_wmo", "CD_MUN"]
    )
)

,codigo_wmo,uf,latitude,longitude,CD_MUN,NM_MUN,SIGLA_UF
265,A857,SC,-26.776667,-53.504444,4204905,Descanso,SC
345,A857,SC,-26.776389,-53.504167,4204905,Descanso,SC
301,A857,SC,-26.786111,-53.513889,4217204,São Miguel do Oeste,SC
356,A857,SC,-26.786389,-53.514167,4217204,São Miguel do Oeste,SC
294,A895,SC,-27.085311,-52.635711,4204202,Chapecó,SC
279,A895,SC,-27.955278,-52.635556,4314779,Pontão,RS
85,A911,MT,-10.165833,-59.451111,5101407,Aripuanã,MT
139,A911,MT,-13.303889,-58.763333,5107875,Sapezal,MT


In [43]:
wmos_investigar = ["A857", "A895", "A911"]

historico_wmos = (
    inmet_diario[
        inmet_diario["codigo_wmo"].isin(wmos_investigar)
    ][
        [
            "codigo_wmo",
            "data",
            "ano",
            "uf",
            "regiao",
            "latitude",
            "longitude",
            "status_dados",
            "completude_pct"
        ]
    ]
    .sort_values(
        ["codigo_wmo", "data"]
    )
)

display(historico_wmos.head(30))

,codigo_wmo,data,ano,uf,regiao,latitude,longitude,status_dados,completude_pct
217536,A857,2019-01-01,2019,SC,sul,-26.776667,-53.504444,dados_completamente_captados,100.0
217627,A857,2019-01-02,2019,SC,sul,-26.776667,-53.504444,dados_completamente_captados,100.0
217718,A857,2019-01-03,2019,SC,sul,-26.776667,-53.504444,dados_completamente_captados,100.0
217809,A857,2019-01-04,2019,SC,sul,-26.776667,-53.504444,dados_completamente_captados,100.0
217900,A857,2019-01-05,2019,SC,sul,-26.776667,-53.504444,dados_completamente_captados,100.0
217991,A857,2019-01-06,2019,SC,sul,-26.776667,-53.504444,dados_completamente_captados,100.0
218082,A857,2019-01-07,2019,SC,sul,-26.776667,-53.504444,dados_completamente_captados,100.0
218173,A857,2019-01-08,2019,SC,sul,-26.776667,-53.504444,dados_completamente_captados,100.0
218264,A857,2019-01-09,2019,SC,sul,-26.776667,-53.504444,dados_completamente_captados,100.0
218355,A857,2019-01-10,2019,SC,sul,-26.776667,-53.504444,dados_completamente_captados,100.0


In [44]:
historico_coords_resumo = (
    historico_wmos
    .groupby(
        [
            "codigo_wmo",
            "uf",
            "regiao",
            "latitude",
            "longitude"
        ],
        as_index=False
    )
    .agg(
        primeira_data=("data", "min"),
        ultima_data=("data", "max"),
        quantidade_dias=("data", "count"),
        completude_media_pct=("completude_pct", "mean")
    )
    .sort_values(
        ["codigo_wmo", "primeira_data"]
    )
)

display(historico_coords_resumo)

,codigo_wmo,uf,regiao,latitude,longitude,primeira_data,ultima_data,quantidade_dias,completude_media_pct
2,A857,SC,sul,-26.776667,-53.504444,2019-01-01,2019-12-31,365,100.0
1,A857,SC,sul,-26.786111,-53.513889,2020-01-01,2021-12-31,731,100.0
3,A857,SC,sul,-26.776389,-53.504167,2022-01-01,2022-12-31,365,100.0
0,A857,SC,sul,-26.786389,-53.514167,2023-01-01,2024-12-31,731,100.0
4,A895,SC,sul,-27.955278,-52.635556,2019-02-19,2019-12-31,316,100.0
5,A895,SC,sul,-27.085311,-52.635711,2020-01-01,2024-12-31,1827,100.0
7,A911,MT,centro_oeste,-10.165833,-59.451111,2019-01-01,2019-12-31,365,100.0
6,A911,MT,centro_oeste,-13.303889,-58.763333,2020-01-01,2024-12-31,1827,100.0


In [45]:
display(
    historico_coords_resumo[
        historico_coords_resumo["codigo_wmo"] == "A895"
    ]
)

,codigo_wmo,uf,regiao,latitude,longitude,primeira_data,ultima_data,quantidade_dias,completude_media_pct
4,A895,SC,sul,-27.955278,-52.635556,2019-02-19,2019-12-31,316,100.0
5,A895,SC,sul,-27.085311,-52.635711,2020-01-01,2024-12-31,1827,100.0


In [46]:
mapa_wmo_coords = (
    estacoes_municipios[
        [
            "codigo_wmo",
            "latitude",
            "longitude",
            "CD_MUN",
            "NM_MUN",
            "SIGLA_UF"
        ]
    ]
    .drop_duplicates()
)

historico_coords_resumo = historico_coords_resumo.merge(
    mapa_wmo_coords,
    on=[
        "codigo_wmo",
        "latitude",
        "longitude"
    ],
    how="left"
)

display(
    historico_coords_resumo[
        historico_coords_resumo["codigo_wmo"].isin(
            wmos_investigar
        )
    ]
)

,codigo_wmo,uf,regiao,latitude,longitude,primeira_data,ultima_data,quantidade_dias,completude_media_pct,CD_MUN,NM_MUN,SIGLA_UF
0,A857,SC,sul,-26.776667,-53.504444,2019-01-01,2019-12-31,365,100.0,4204905,Descanso,SC
1,A857,SC,sul,-26.786111,-53.513889,2020-01-01,2021-12-31,731,100.0,4217204,São Miguel do Oeste,SC
2,A857,SC,sul,-26.776389,-53.504167,2022-01-01,2022-12-31,365,100.0,4204905,Descanso,SC
3,A857,SC,sul,-26.786389,-53.514167,2023-01-01,2024-12-31,731,100.0,4217204,São Miguel do Oeste,SC
4,A895,SC,sul,-27.955278,-52.635556,2019-02-19,2019-12-31,316,100.0,4314779,Pontão,RS
5,A895,SC,sul,-27.085311,-52.635711,2020-01-01,2024-12-31,1827,100.0,4204202,Chapecó,SC
6,A911,MT,centro_oeste,-10.165833,-59.451111,2019-01-01,2019-12-31,365,100.0,5101407,Aripuanã,MT
7,A911,MT,centro_oeste,-13.303889,-58.763333,2020-01-01,2024-12-31,1827,100.0,5107875,Sapezal,MT


In [47]:
estacoes_ano = (
    inmet_diario
    .groupby(
        [
            "codigo_wmo",
            "ano",
            "uf",
            "regiao"
        ],
        as_index=False
    )
    .agg(
        latitude=("latitude", "median"),
        longitude=("longitude", "median"),
        dias_registrados=("data", "count"),
        primeira_data=("data", "min"),
        ultima_data=("data", "max")
    )
)

print("Dimensão da tabela estação × ano:")
print(estacoes_ano.shape)

print("\nQuantidade de WMO:")
print(estacoes_ano["codigo_wmo"].nunique())

print("\nAnos:")
print(sorted(estacoes_ano["ano"].unique()))

display(estacoes_ano.head(10))

Dimensão da tabela estação × ano:
(1155, 9)

Quantidade de WMO:
209

Anos:
[np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)]


,codigo_wmo,ano,uf,regiao,latitude,longitude,dias_registrados,primeira_data,ultima_data
0,A001,2019,DF,centro_oeste,-15.789343,-47.925756,365,2019-01-01,2019-12-31
1,A001,2020,DF,centro_oeste,-15.789343,-47.925756,366,2020-01-01,2020-12-31
2,A001,2021,DF,centro_oeste,-15.789444,-47.925833,365,2021-01-01,2021-12-31
3,A001,2022,DF,centro_oeste,-15.789444,-47.925833,365,2022-01-01,2022-12-31
4,A001,2023,DF,centro_oeste,-15.789444,-47.925833,365,2023-01-01,2023-12-31
5,A001,2024,DF,centro_oeste,-15.789444,-47.925833,366,2024-01-01,2024-12-31
6,A002,2019,GO,centro_oeste,-16.642841,-49.220222,365,2019-01-01,2019-12-31
7,A002,2020,GO,centro_oeste,-16.642841,-49.220222,366,2020-01-01,2020-12-31
8,A002,2021,GO,centro_oeste,-16.642841,-49.220222,365,2021-01-01,2021-12-31
9,A002,2022,GO,centro_oeste,-16.642841,-49.220222,365,2022-01-01,2022-12-31


In [48]:
estacoes_ano_geo = gpd.GeoDataFrame(
    estacoes_ano,
    geometry=gpd.points_from_xy(
        estacoes_ano["longitude"],
        estacoes_ano["latitude"]
    ),
    crs="EPSG:4674"
)

print("CRS:")
print(estacoes_ano_geo.crs)

print("\nQuantidade de pontos estação-ano:")
print(len(estacoes_ano_geo))

CRS:
EPSG:4674

Quantidade de pontos estação-ano:
1155


In [49]:
estacoes_ano_municipios = gpd.sjoin(
    estacoes_ano_geo,
    malha_join,
    how="left",
    predicate="within"
)

print("Pontos estação-ano sem município:")
print(estacoes_ano_municipios["CD_MUN"].isna().sum())

print("\nMunicípios atingidos:")
print(estacoes_ano_municipios["CD_MUN"].nunique())

Pontos estação-ano sem município:
0

Municípios atingidos:
205


In [50]:
estacoes_ano_municipios["uf_confere"] = (
    estacoes_ano_municipios["uf"]
    == estacoes_ano_municipios["SIGLA_UF"]
)

print("Registros estação-ano com UF divergente:")
print(
    (~estacoes_ano_municipios["uf_confere"]).sum()
)

Registros estação-ano com UF divergente:
1


In [51]:
display(
    estacoes_ano_municipios[
        ~estacoes_ano_municipios["uf_confere"]
    ][
        [
            "codigo_wmo",
            "ano",
            "uf",
            "latitude",
            "longitude",
            "CD_MUN",
            "NM_MUN",
            "SIGLA_UF"
        ]
    ]
)

,codigo_wmo,ano,uf,latitude,longitude,CD_MUN,NM_MUN,SIGLA_UF
858,A895,2019,SC,-27.955278,-52.635556,4314779,Pontão,RS


In [52]:
estacoes_ano_municipios["qualidade_geografica"] = np.where(
    estacoes_ano_municipios["uf_confere"],
    "ok",
    "uf_divergente"
)

print("Qualidade geográfica:")
print(
    estacoes_ano_municipios["qualidade_geografica"]
    .value_counts()
)

Qualidade geográfica:
qualidade_geografica
ok               1154
uf_divergente       1
Name: count, dtype: int64


In [53]:
display(
    estacoes_ano_municipios[
        estacoes_ano_municipios["qualidade_geografica"] != "ok"
    ][
        [
            "codigo_wmo",
            "ano",
            "uf",
            "latitude",
            "longitude",
            "CD_MUN",
            "NM_MUN",
            "SIGLA_UF",
            "qualidade_geografica"
        ]
    ]
)

,codigo_wmo,ano,uf,latitude,longitude,CD_MUN,NM_MUN,SIGLA_UF,qualidade_geografica
858,A895,2019,SC,-27.955278,-52.635556,4314779,Pontão,RS,uf_divergente


In [54]:
estacoes_ano_municipios = (
    estacoes_ano_municipios
    .rename(
        columns={
            "CD_MUN": "codigo_ibge",
            "NM_MUN": "municipio",
            "SIGLA_UF": "uf_ibge",
            "NM_REGIA": "regiao_ibge"
        }
    )
)

estacoes_ano_municipios["codigo_ibge"] = (
    estacoes_ano_municipios["codigo_ibge"]
    .astype(str)
)

display(
    estacoes_ano_municipios[
        [
            "codigo_wmo",
            "ano",
            "uf",
            "codigo_ibge",
            "municipio",
            "uf_ibge",
            "regiao_ibge",
            "qualidade_geografica"
        ]
    ].head(10)
)

,codigo_wmo,ano,uf,codigo_ibge,municipio,uf_ibge,regiao_ibge,qualidade_geografica
0,A001,2019,DF,5300108,Brasília,DF,Centro-oeste,ok
1,A001,2020,DF,5300108,Brasília,DF,Centro-oeste,ok
2,A001,2021,DF,5300108,Brasília,DF,Centro-oeste,ok
3,A001,2022,DF,5300108,Brasília,DF,Centro-oeste,ok
4,A001,2023,DF,5300108,Brasília,DF,Centro-oeste,ok
5,A001,2024,DF,5300108,Brasília,DF,Centro-oeste,ok
6,A002,2019,GO,5208707,Goiânia,GO,Centro-oeste,ok
7,A002,2020,GO,5208707,Goiânia,GO,Centro-oeste,ok
8,A002,2021,GO,5208707,Goiânia,GO,Centro-oeste,ok
9,A002,2022,GO,5208707,Goiânia,GO,Centro-oeste,ok


In [55]:
variaveis_binarias = [
    "choveu",
    "chuva_forte",
    "temperatura_extrema_alta",
    "temperatura_extrema_baixa",
    "ventania_extrema",
    "vento_forte",
    "umidade_muito_alta",
    "umidade_baixa"
]

for coluna in variaveis_binarias:
    print(
        f"{coluna}:",
        sorted(
            inmet_diario[coluna]
            .dropna()
            .astype(str)
            .unique()
        )
    )

choveu: ['dados_nao_captados', 'nao', 'sim']
chuva_forte: ['dados_nao_captados', 'nao']
temperatura_extrema_alta: ['dados_nao_captados', 'nao', 'sim']
temperatura_extrema_baixa: ['dados_nao_captados', 'nao', 'sim']
ventania_extrema: ['dados_nao_captados', 'nao', 'sim']
vento_forte: ['dados_nao_captados', 'nao', 'sim']
umidade_muito_alta: ['dados_nao_captados', 'nao', 'sim']
umidade_baixa: ['dados_nao_captados', 'nao', 'sim']


In [56]:
mapa_binario = {
    "sim": 1,
    "nao": 0,
    "dados_nao_captados": np.nan
}

for coluna in variaveis_binarias:
    inmet_diario[f"{coluna}_bin"] = (
        inmet_diario[coluna]
        .map(mapa_binario)
    )

print("Conversão concluída.")

for coluna in variaveis_binarias:
    print(
        f"{coluna}_bin:",
        inmet_diario[f"{coluna}_bin"]
        .value_counts(dropna=False)
        .to_dict()
    )

Conversão concluída.
choveu_bin: {0.0: 411167, nan: 9134, 1.0: 746}
chuva_forte_bin: {0.0: 411913, nan: 9134}
temperatura_extrema_alta_bin: {0.0: 327554, nan: 57296, 1.0: 36197}
temperatura_extrema_baixa_bin: {0.0: 326793, nan: 57299, 1.0: 36955}
ventania_extrema_bin: {0.0: 346763, nan: 69745, 1.0: 4539}
vento_forte_bin: {0.0: 235540, 1.0: 115762, nan: 69745}
umidade_muito_alta_bin: {0.0: 325408, nan: 65065, 1.0: 30574}
umidade_baixa_bin: {0.0: 350891, nan: 65065, 1.0: 5091}


In [57]:
inmet_estacao_ano = (
    inmet_diario
    .groupby(
        ["codigo_wmo", "ano"],
        as_index=False
    )
    .agg(
        precipitacao_anual_mm=(
            "precipitacao_total_mm",
            "sum"
        ),

        temperatura_media_anual_c=(
            "temperatura_media_c",
            "mean"
        ),

        temperatura_maxima_media_c=(
            "temperatura_maxima_c",
            "mean"
        ),

        temperatura_minima_media_c=(
            "temperatura_minima_c",
            "mean"
        ),

        umidade_media_anual_pct=(
            "umidade_media_pct",
            "mean"
        ),

        radiacao_anual_mj_m2=(
            "radiacao_total_mj_m2",
            "sum"
        ),

        rajada_maxima_anual_ms=(
            "rajada_maxima_ms",
            "max"
        ),

        velocidade_vento_media_anual_ms=(
            "velocidade_vento_media_ms",
            "mean"
        ),

        dias_com_chuva=(
            "choveu_bin",
            "sum"
        ),

        dias_chuva_forte=(
            "chuva_forte_bin",
            "sum"
        ),

        dias_calor_extremo=(
            "temperatura_extrema_alta_bin",
            "sum"
        ),

        dias_frio_extremo=(
            "temperatura_extrema_baixa_bin",
            "sum"
        ),

        dias_ventania_extrema=(
            "ventania_extrema_bin",
            "sum"
        ),

        dias_vento_forte=(
            "vento_forte_bin",
            "sum"
        ),

        dias_umidade_muito_alta=(
            "umidade_muito_alta_bin",
            "sum"
        ),

        dias_umidade_baixa=(
            "umidade_baixa_bin",
            "sum"
        ),

        completude_media_pct=(
            "completude_pct",
            "mean"
        ),

        dias_observados=(
            "data",
            "count"
        )
    )
)

print("Dimensão da base climática estação × ano:")
print(inmet_estacao_ano.shape)

print("\nPeríodo:")
print(
    inmet_estacao_ano["ano"]
    .min(),
    "até",
    inmet_estacao_ano["ano"]
    .max()
)

display(inmet_estacao_ano.head(10))

Dimensão da base climática estação × ano:
(1155, 20)

Período:
2019 até 2024


,codigo_wmo,ano,precipitacao_anual_mm,temperatura_media_anual_c,temperatura_maxima_media_c,temperatura_minima_media_c,umidade_media_anual_pct,radiacao_anual_mj_m2,rajada_maxima_anual_ms,velocidade_vento_media_anual_ms,dias_com_chuva,dias_chuva_forte,dias_calor_extremo,dias_frio_extremo,dias_ventania_extrema,dias_vento_forte,dias_umidade_muito_alta,dias_umidade_baixa,completude_media_pct,dias_observados
0,A001,2019,0.0,21.976633,28.050685,16.936438,64.033021,7119.7316,16.6,2.188658,0.0,0.0,0.0,3.0,0.0,71.0,1.0,14.0,100.0,365
1,A001,2020,0.0,21.348964,27.234153,16.533880,67.105191,6941.9034,15.3,2.250376,0.0,0.0,4.0,7.0,0.0,74.0,14.0,11.0,100.0,366
2,A001,2021,0.0,21.223166,27.307123,16.214795,65.541270,6958.2657,15.2,2.104258,0.0,0.0,1.0,11.0,0.0,70.0,7.0,17.0,100.0,365
3,A001,2022,0.0,21.246632,27.452055,16.121370,63.533219,7003.6890,15.7,2.248986,0.0,0.0,0.0,7.0,0.0,63.0,6.0,9.0,100.0,365
4,A001,2023,0.0,21.956853,28.311781,16.721918,64.850454,7047.3686,15.3,2.010873,0.0,0.0,0.0,2.0,0.0,63.0,1.0,1.0,100.0,365
5,A001,2024,0.0,21.976389,28.198361,16.866120,67.236501,6790.8167,15.3,2.085332,0.0,0.0,5.0,2.0,0.0,56.0,19.0,21.0,100.0,366
6,A002,2019,0.0,24.342137,32.147945,18.403836,61.855683,6668.4732,13.9,1.084918,0.0,0.0,58.0,3.0,0.0,62.0,0.0,7.0,100.0,365
7,A002,2020,0.0,24.003124,31.560656,18.115574,62.498240,6608.9677,14.7,1.063376,0.0,0.0,45.0,4.0,0.0,65.0,0.0,12.0,100.0,366
8,A002,2021,0.0,23.728584,31.432877,17.798082,62.831568,6427.6880,13.4,0.946078,0.0,0.0,37.0,11.0,0.0,38.0,3.0,5.0,100.0,365
9,A002,2022,0.0,23.668447,31.470959,17.640548,61.392352,6622.4655,14.3,1.000518,0.0,0.0,30.0,7.0,0.0,45.0,1.0,8.0,100.0,365


In [58]:
print("Duplicidades codigo_wmo + ano:")
print(
    inmet_estacao_ano[
        ["codigo_wmo", "ano"]
    ]
    .duplicated()
    .sum()
)

print("\nValores ausentes por coluna:")
display(
    inmet_estacao_ano
    .isna()
    .sum()
    .sort_values(ascending=False)
)

Duplicidades codigo_wmo + ano:
0

Valores ausentes por coluna:


velocidade_vento_media_anual_ms    15
umidade_media_anual_pct            14
rajada_maxima_anual_ms             14
temperatura_media_anual_c           3
temperatura_maxima_media_c          3
temperatura_minima_media_c          3
codigo_wmo                          0
dias_frio_extremo                   0
completude_media_pct                0
dias_umidade_baixa                  0
dias_umidade_muito_alta             0
dias_vento_forte                    0
dias_ventania_extrema               0
dias_com_chuva                      0
dias_calor_extremo                  0
dias_chuva_forte                    0
ano                                 0
radiacao_anual_mj_m2                0
precipitacao_anual_mm               0
dias_observados                     0
dtype: int64

In [59]:
print("Tipo da precipitação:")
print(inmet_diario["precipitacao_total_mm"].dtype)

print("\nQuantidade total de registros:")
print(len(inmet_diario))

print("\nValores ausentes:")
print(inmet_diario["precipitacao_total_mm"].isna().sum())

print("\nValores iguais a zero:")
print((inmet_diario["precipitacao_total_mm"] == 0).sum())

print("\nValores maiores que zero:")
print((inmet_diario["precipitacao_total_mm"] > 0).sum())

print("\nEstatísticas:")
display(
    inmet_diario["precipitacao_total_mm"].describe()
)

Tipo da precipitação:
float64

Quantidade total de registros:
421047

Valores ausentes:
9134

Valores iguais a zero:
411167

Valores maiores que zero:
746

Estatísticas:


count    411913.000000
mean          0.003925
std           0.121953
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          11.800000
Name: precipitacao_total_mm, dtype: float64

In [60]:
display(
    inmet_diario[
        [
            "codigo_wmo",
            "data",
            "uf",
            "precipitacao_total_mm",
            "choveu",
            "chuva_forte",
            "completude_pct",
            "status_dados"
        ]
    ]
    .sort_values(
        "precipitacao_total_mm",
        ascending=False
    )
    .head(20)
)

,codigo_wmo,data,uf,precipitacao_total_mm,choveu,chuva_forte,completude_pct,status_dados
373401,A845,2023-08-10,SC,11.8,sim,nao,100.0,dados_completamente_captados
148660,A930,2023-01-07,MT,11.8,sim,nao,100.0,dados_completamente_captados
148468,A930,2023-01-05,MT,11.2,sim,nao,100.0,dados_completamente_captados
400006,A845,2024-05-19,SC,9.2,sim,nao,100.0,dados_completamente_captados
377866,A845,2023-09-26,SC,9.0,sim,nao,100.0,dados_completamente_captados
376156,A845,2023-09-08,SC,8.8,sim,nao,100.0,dados_completamente_captados
83325,A703,2021-01-24,MS,8.8,sim,nao,100.0,dados_completamente_captados
148564,A930,2023-01-06,MT,8.8,sim,nao,100.0,dados_completamente_captados
148756,A930,2023-01-08,MT,7.4,sim,nao,100.0,dados_completamente_captados
392287,A845,2024-02-26,SC,7.2,sim,nao,100.0,dados_completamente_captados


In [61]:
display(
    inmet_diario[
        (inmet_diario["codigo_wmo"] == "A001")
        &
        (inmet_diario["ano"] == 2019)
    ][
        [
            "data",
            "precipitacao_total_mm",
            "choveu",
            "chuva_forte",
            "temperatura_media_c",
            "completude_pct",
            "status_dados"
        ]
    ]
    .head(30)
)

,data,precipitacao_total_mm,choveu,chuva_forte,temperatura_media_c,completude_pct,status_dados
0,2019-01-01,0.0,nao,nao,20.520833,100.0,dados_completamente_captados
111,2019-01-02,0.0,nao,nao,22.716667,100.0,dados_completamente_captados
222,2019-01-03,0.0,nao,nao,24.341667,100.0,dados_completamente_captados
333,2019-01-04,0.0,nao,nao,23.425000,100.0,dados_completamente_captados
444,2019-01-05,0.0,nao,nao,20.866667,100.0,dados_completamente_captados
555,2019-01-06,0.0,nao,nao,22.220833,100.0,dados_completamente_captados
666,2019-01-07,0.0,nao,nao,22.670833,100.0,dados_completamente_captados
777,2019-01-08,0.0,nao,nao,23.425000,100.0,dados_completamente_captados
888,2019-01-09,0.0,nao,nao,22.762500,100.0,dados_completamente_captados
999,2019-01-10,0.0,nao,nao,21.200000,100.0,dados_completamente_captados


In [62]:
print("A001 2019 - precipitação:")
print(
    inmet_diario.loc[
        (inmet_diario["codigo_wmo"] == "A001")
        &
        (inmet_diario["ano"] == 2019),
        "precipitacao_total_mm"
    ].describe()
)

A001 2019 - precipitação:
count    365.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: precipitacao_total_mm, dtype: float64


In [63]:
print("Tabela choveu × precipitação > 0:")

display(
    pd.crosstab(
        inmet_diario["choveu"],
        inmet_diario["precipitacao_total_mm"] > 0,
        margins=True
    )
)

Tabela choveu × precipitação > 0:


precipitacao_total_mm,False,True,All
choveu,,,
dados_nao_captados,9134,0,9134
nao,411167,0,411167
sim,0,746,746
All,420301,746,421047


In [64]:
INMET_PROCESSED_DIR = (
    BASE_DIR
    / "data"
    / "raw"
    / "INMET_LIMPO"
    / "databases_processed"
)

print("Processed existe?")
print(INMET_PROCESSED_DIR.exists())

print("\nConteúdo:")
for item in sorted(INMET_PROCESSED_DIR.iterdir()):
    print(
        "[PASTA]" if item.is_dir() else "[ARQUIVO]",
        item.name
    )

Processed existe?
True

Conteúdo:
[PASTA] INMET


In [65]:
arquivos_processed = sorted(
    INMET_PROCESSED_DIR.rglob("*.csv")
)

print("Quantidade de CSVs processed:")
print(len(arquivos_processed))

print("\nPrimeiros 20 arquivos:")
for arquivo in arquivos_processed[:20]:
    print(arquivo.relative_to(INMET_PROCESSED_DIR))

Quantidade de CSVs processed:
1195

Primeiros 20 arquivos:
INMET\2019\2019_A001_0001.csv
INMET\2019\2019_A002_0006.csv
INMET\2019\2019_A003_0007.csv
INMET\2019\2019_A005_0008.csv
INMET\2019\2019_A011_0009.csv
INMET\2019\2019_A012_0010.csv
INMET\2019\2019_A013_0011.csv
INMET\2019\2019_A014_0012.csv
INMET\2019\2019_A015_0013.csv
INMET\2019\2019_A016_0014.csv
INMET\2019\2019_A017_0015.csv
INMET\2019\2019_A022_0016.csv
INMET\2019\2019_A023_0017.csv
INMET\2019\2019_A024_0018.csv
INMET\2019\2019_A025_0019.csv
INMET\2019\2019_A026_0020.csv
INMET\2019\2019_A027_0021.csv
INMET\2019\2019_A028_0022.csv
INMET\2019\2019_A029_0023.csv
INMET\2019\2019_A031_0024.csv


In [66]:
if arquivos_processed:

    exemplo_processed = pd.read_csv(
        arquivos_processed[0],
        nrows=5
    )

    print("Arquivo:")
    print(
        arquivos_processed[0]
        .relative_to(INMET_PROCESSED_DIR)
    )

    print("\nDimensão da amostra:")
    print(exemplo_processed.shape)

    print("\nColunas:")
    print(exemplo_processed.columns.tolist())

    display(exemplo_processed.head())

Arquivo:
INMET\2019\2019_A001_0001.csv

Dimensão da amostra:
(5, 41)

Colunas:
['data', 'hora_utc', 'precipitacao_mm', 'pressao_mb', 'pressao_max_mb', 'pressao_min_mb', 'radiacao_kj_m2', 'temperatura_c', 'ponto_orvalho_c', 'temperatura_max_c', 'temperatura_min_c', 'ponto_orvalho_max_c', 'ponto_orvalho_min_c', 'umidade_max_pct', 'umidade_min_pct', 'umidade_pct', 'direcao_vento_graus', 'rajada_max_ms', 'velocidade_vento_ms', 'uf', 'regiao', 'codigo_wmo', 'estacao', 'latitude', 'longitude', 'altitude_m', 'precipitacao_mm_status', 'pressao_mb_status', 'radiacao_kj_m2_status', 'temperatura_c_status', 'ponto_orvalho_c_status', 'temperatura_max_c_status', 'temperatura_min_c_status', 'ponto_orvalho_max_c_status', 'ponto_orvalho_min_c_status', 'umidade_pct_status', 'umidade_max_pct_status', 'umidade_min_pct_status', 'direcao_vento_graus_status', 'rajada_max_ms_status', 'velocidade_vento_ms_status']


,data,hora_utc,precipitacao_mm,pressao_mb,pressao_max_mb,pressao_min_mb,radiacao_kj_m2,temperatura_c,ponto_orvalho_c,temperatura_max_c,...,temperatura_max_c_status,temperatura_min_c_status,ponto_orvalho_max_c_status,ponto_orvalho_min_c_status,umidade_pct_status,umidade_max_pct_status,umidade_min_pct_status,direcao_vento_graus_status,rajada_max_ms_status,velocidade_vento_ms_status
0,2019-01-01,0000 UTC,NaN,887.0,887.0,886.6,NaN,18.5,17.0,18.7,...,captado,captado,captado,captado,captado,captado,captado,captado,captado,captado
1,2019-01-01,0100 UTC,0.0,888.1,888.1,887.0,NaN,18.4,17.1,18.5,...,captado,captado,captado,captado,captado,captado,captado,captado,captado,captado
2,2019-01-01,0200 UTC,0.0,888.2,888.3,888.1,NaN,18.5,17.3,18.6,...,captado,captado,captado,captado,captado,captado,captado,captado,captado,captado
3,2019-01-01,0300 UTC,NaN,887.6,888.2,887.6,NaN,18.4,17.1,18.7,...,captado,captado,captado,captado,captado,captado,captado,captado,captado,captado
4,2019-01-01,0400 UTC,0.0,887.0,887.6,887.0,NaN,17.9,16.7,18.4,...,captado,captado,captado,captado,captado,captado,captado,captado,captado,captado


In [67]:
chuva_por_ano = (
    inmet_diario
    .assign(
        chuva_positiva=inmet_diario[
            "precipitacao_total_mm"
        ].gt(0)
    )
    .groupby("ano")
    .agg(
        registros=("codigo_wmo", "size"),
        chuva_positiva=("chuva_positiva", "sum"),
        precipitacao_max_mm=(
            "precipitacao_total_mm",
            "max"
        ),
        precipitacao_media_mm=(
            "precipitacao_total_mm",
            "mean"
        )
    )
)

display(chuva_por_ano)

,registros,chuva_positiva,precipitacao_max_mm,precipitacao_media_mm
ano,,,,
2019,75343,0,0.0,0.000000
2020,74298,95,7.0,0.004106
2021,65335,41,8.8,0.001559
2022,67914,50,5.8,0.001369
2023,69715,308,11.8,0.008584
2024,68442,252,9.2,0.008191


In [68]:
chuva_por_uf = (
    inmet_diario
    .assign(
        chuva_positiva=inmet_diario[
            "precipitacao_total_mm"
        ].gt(0)
    )
    .groupby("uf")
    .agg(
        registros=("codigo_wmo", "size"),
        chuva_positiva=("chuva_positiva", "sum"),
        precipitacao_max_mm=(
            "precipitacao_total_mm",
            "max"
        )
    )
    .sort_values(
        "chuva_positiva",
        ascending=False
    )
)

display(chuva_por_uf)

,registros,chuva_positiva,precipitacao_max_mm
uf,,,
SC,52514,581,11.8
MT,75708,124,11.8
MS,73798,41,8.8
DF,10960,0,0.0
GO,56992,0,0.0
PR,54801,0,0.0
RS,96274,0,0.0


In [69]:
arquivos_a001_2019 = list(
    INMET_PROCESSED_DIR.rglob("2019_A001_*.csv")
)

print("Arquivos encontrados:")
for arquivo in arquivos_a001_2019:
    print(
        arquivo.relative_to(INMET_PROCESSED_DIR)
    )

Arquivos encontrados:
INMET\2019\2019_A001_0001.csv


In [70]:
arquivo_a001_2019 = arquivos_a001_2019[0]

a001_2019_processed = pd.read_csv(
    arquivo_a001_2019
)

a001_2019_processed["data"] = pd.to_datetime(
    a001_2019_processed["data"],
    errors="coerce"
)

a001_2019_processed["precipitacao_mm"] = pd.to_numeric(
    a001_2019_processed["precipitacao_mm"],
    errors="coerce"
)

print("Dimensão:")
print(a001_2019_processed.shape)

print("\nPeríodo:")
print(a001_2019_processed["data"].min())
print(a001_2019_processed["data"].max())

print("\nPrecipitação horária:")
print(
    a001_2019_processed[
        "precipitacao_mm"
    ].describe()
)

Dimensão:
(8760, 41)

Período:
2019-01-01 00:00:00
2019-12-31 00:00:00

Precipitação horária:
count    8259.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: precipitacao_mm, dtype: float64


In [71]:
print("Valores ausentes:")
print(
    a001_2019_processed[
        "precipitacao_mm"
    ].isna().sum()
)

print("\nValores = 0:")
print(
    (
        a001_2019_processed[
            "precipitacao_mm"
        ] == 0
    ).sum()
)

print("\nValores > 0:")
print(
    (
        a001_2019_processed[
            "precipitacao_mm"
        ] > 0
    ).sum()
)

print("\nMaior precipitação horária:")
print(
    a001_2019_processed[
        "precipitacao_mm"
    ].max()
)

print("\nSoma de toda a precipitação horária:")
print(
    a001_2019_processed[
        "precipitacao_mm"
    ].sum(
        min_count=1
    )
)

Valores ausentes:
501

Valores = 0:
8259

Valores > 0:
0

Maior precipitação horária:
0.0

Soma de toda a precipitação horária:
0.0


In [72]:
display(
    a001_2019_processed[
        a001_2019_processed[
            "precipitacao_mm"
        ] > 0
    ][
        [
            "data",
            "hora_utc",
            "precipitacao_mm",
            "precipitacao_mm_status"
        ]
    ]
    .head(30)
)

,data,hora_utc,precipitacao_mm,precipitacao_mm_status


# 6. Retorno à fonte bruta original do INMET

As validações realizadas sobre as camadas Processed e Curated identificaram
uma inconsistência relevante na variável de precipitação.

Para garantir rastreabilidade e confiabilidade na construção da dimensão
climática, a partir desta etapa os dados serão novamente processados
diretamente dos arquivos históricos originais do INMET.

Fonte utilizada:

`data/raw/inmet/{ano}/arquivo_inmet_{ano}.zip`

Período: 2019–2024.

Os arquivos originais permanecerão inalterados. A pasta `INMET_LIMPO`
será mantida apenas para diagnóstico e comparação, não sendo utilizada
como fonte oficial da integração final.

In [73]:
from pathlib import Path
import zipfile

INMET_RAW_DIR = (
    BASE_DIR
    / "data"
    / "raw"
    / "inmet"
)

ANOS_PROJETO = list(range(2019, 2025))

print("Diretório INMET bruto:")
print(INMET_RAW_DIR)

print("\nDiretório existe?")
print(INMET_RAW_DIR.exists())

print("\nAnos esperados:")
print(ANOS_PROJETO)

Diretório INMET bruto:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\raw\inmet

Diretório existe?
True

Anos esperados:
[2019, 2020, 2021, 2022, 2023, 2024]


In [74]:
from pathlib import Path
import zipfile

INMET_RAW_DIR = (
    BASE_DIR
    / "data"
    / "raw"
    / "inmet"
)

ANOS_PROJETO = list(range(2019, 2025))

print("Diretório INMET bruto:")
print(INMET_RAW_DIR)

print("\nDiretório existe?")
print(INMET_RAW_DIR.exists())

print("\nAnos esperados:")
print(ANOS_PROJETO)

Diretório INMET bruto:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\raw\inmet

Diretório existe?
True

Anos esperados:
[2019, 2020, 2021, 2022, 2023, 2024]


In [75]:
zips_por_ano = {}

for ano in ANOS_PROJETO:

    pasta_ano = INMET_RAW_DIR / str(ano)

    candidatos = [
        arquivo
        for arquivo in pasta_ano.iterdir()
        if arquivo.is_file()
        and zipfile.is_zipfile(arquivo)
    ]

    print(f"\n{ano}:")
    print(f"ZIPs encontrados: {len(candidatos)}")

    for arquivo in candidatos:
        print(
            " -",
            arquivo.name,
            f"({arquivo.stat().st_size / 1024**2:.2f} MB)"
        )

    if len(candidatos) == 1:
        zips_por_ano[ano] = candidatos[0]


2019:
ZIPs encontrados: 1
 - arquivo_inmet_2019.zip (112.11 MB)

2020:
ZIPs encontrados: 1
 - arquivo_inmet_2020.zip (98.85 MB)

2021:
ZIPs encontrados: 1
 - arquivo_inmet_2021.zip (76.84 MB)

2022:
ZIPs encontrados: 1
 - arquivo_inmet_2022.zip (86.18 MB)

2023:
ZIPs encontrados: 1
 - arquivo_inmet_2023.zip (102.11 MB)

2024:
ZIPs encontrados: 1
 - arquivo_inmet_2024.zip (98.01 MB)


In [76]:
print("\nAnos com ZIP identificado:")
print(sorted(zips_por_ano.keys()))


Anos com ZIP identificado:
[2019, 2020, 2021, 2022, 2023, 2024]


In [77]:
resumo_zips = []

for ano, caminho_zip in sorted(zips_por_ano.items()):

    with zipfile.ZipFile(caminho_zip) as z:

        membros = z.namelist()

        csvs = [
            nome
            for nome in membros
            if nome.lower().endswith(".csv")
        ]

        resumo_zips.append(
            {
                "ano": ano,
                "arquivo_zip": caminho_zip.name,
                "itens_zip": len(membros),
                "arquivos_csv": len(csvs)
            }
        )

        print("=" * 70)
        print("ANO:", ano)
        print("ZIP:", caminho_zip.name)
        print("Itens:", len(membros))
        print("CSVs:", len(csvs))

        print("\nPrimeiros 5 CSVs:")

        for nome in csvs[:5]:
            print(" -", nome)

ANO: 2019
ZIP: arquivo_inmet_2019.zip
Itens: 590
CSVs: 589

Primeiros 5 CSVs:
 - 2019/INMET_CO_DF_A001_BRASILIA_01-01-2019_A_31-12-2019.CSV
 - 2019/INMET_CO_DF_A042_BRAZLANDIA_01-01-2019_A_31-12-2019.CSV
 - 2019/INMET_CO_DF_A045_AGUAS EMENDADAS_01-01-2019_A_31-12-2019.CSV
 - 2019/INMET_CO_DF_A046_GAMA (PONTE ALTA)_01-01-2019_A_31-12-2019.CSV
 - 2019/INMET_CO_DF_A047_PARANOA (COOPA-DF)_01-01-2019_A_31-12-2019.CSV
ANO: 2020
ZIP: arquivo_inmet_2020.zip
Itens: 589
CSVs: 589

Primeiros 5 CSVs:
 - INMET_CO_DF_A001_BRASILIA_01-01-2020_A_31-12-2020.CSV
 - INMET_CO_DF_A042_BRAZLANDIA_01-01-2020_A_31-12-2020.CSV
 - INMET_CO_DF_A045_AGUAS EMENDADAS_01-01-2020_A_31-12-2020.CSV
 - INMET_CO_DF_A046_GAMA (PONTE ALTA)_01-01-2020_A_31-12-2020.CSV
 - INMET_CO_DF_A047_PARANOA (COOPA-DF)_01-01-2020_A_31-12-2020.CSV
ANO: 2021
ZIP: arquivo_inmet_2021.zip
Itens: 588
CSVs: 588

Primeiros 5 CSVs:
 - INMET_CO_DF_A001_BRASILIA_01-01-2021_A_31-12-2021.CSV
 - INMET_CO_DF_A042_BRAZLANDIA_01-01-2021_A_31-12-2021.CSV

In [78]:
resumo_zips = pd.DataFrame(resumo_zips)

display(resumo_zips)

,ano,arquivo_zip,itens_zip,arquivos_csv
0,2019,arquivo_inmet_2019.zip,590,589
1,2020,arquivo_inmet_2020.zip,589,589
2,2021,arquivo_inmet_2021.zip,588,588
3,2022,arquivo_inmet_2022.zip,567,567
4,2023,arquivo_inmet_2023.zip,567,567
5,2024,arquivo_inmet_2024.zip,565,565


In [79]:
ZIP_2019 = zips_por_ano[2019]

with zipfile.ZipFile(ZIP_2019) as z:

    csvs_2019 = [
        nome
        for nome in z.namelist()
        if nome.lower().endswith(".csv")
    ]

arquivos_a001 = [
    nome
    for nome in csvs_2019
    if "A001" in nome.upper()
]

print("Arquivos encontrados para A001:")

for nome in arquivos_a001:
    print(nome)

Arquivos encontrados para A001:
2019/INMET_CO_DF_A001_BRASILIA_01-01-2019_A_31-12-2019.CSV


In [80]:
nome_a001 = arquivos_a001[0]

with zipfile.ZipFile(ZIP_2019) as z:

    with z.open(nome_a001) as arquivo:

        for numero in range(15):

            linha = arquivo.readline()

            texto = linha.decode(
                "latin1",
                errors="replace"
            )

            print(
                numero,
                ":",
                texto.strip()
            )

0 : REGI?O:;CO
1 : UF:;DF
2 : ESTAC?O:;BRASILIA
3 : CODIGO (WMO):;A001
4 : LATITUDE:;-15,789343
5 : LONGITUDE:;-47,925756
6 : ALTITUDE:;1160,96
7 : DATA DE FUNDAC?O:;07/05/00
8 : Data;Hora UTC;PRECIPITAÇÃO TOTAL, HORÁRIO (mm);PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB);PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB);PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB);RADIACAO GLOBAL (KJ/m²);TEMPERATURA DO AR - BULBO SECO, HORARIA (°C);TEMPERATURA DO PONTO DE ORVALHO (°C);TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C);TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C);TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C);TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C);UMIDADE REL. MAX. NA HORA ANT. (AUT) (%);UMIDADE REL. MIN. NA HORA ANT. (AUT) (%);UMIDADE RELATIVA DO AR, HORARIA (%);VENTO, DIREÇÃO HORARIA (gr) (° (gr));VENTO, RAJADA MAXIMA (m/s);VENTO, VELOCIDADE HORARIA (m/s);
9 : 2019/01/01;0000 UTC;1;887;887;886,6;;18,5;17;18,7;18,4;17,3;16,9;92;91;91;330;5,3;2;
10 : 2019/01/01;0100 UTC;0;88

In [81]:
import re
import unicodedata
import numpy as np
import pandas as pd


def normalizar_texto(texto):

    texto = str(texto)

    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = "".join(
        caractere
        for caractere in texto
        if not unicodedata.combining(caractere)
    )

    texto = texto.lower()

    texto = re.sub(
        r"[^a-z0-9]+",
        "_",
        texto
    )

    return texto.strip("_")

In [82]:
def inspecionar_arquivo_inmet(
    zip_aberto,
    nome_arquivo,
    max_linhas=20
):

    linhas = []

    with zip_aberto.open(nome_arquivo) as arquivo:

        for _ in range(max_linhas):

            linha = arquivo.readline()

            if not linha:
                break

            linhas.append(
                linha.decode(
                    "latin1",
                    errors="replace"
                ).strip()
            )

    linha_cabecalho = None

    for i, linha in enumerate(linhas):

        linha_norm = normalizar_texto(linha)

        if (
            linha.lower().startswith("data;")
            and "precipitacao" in linha_norm
        ):
            linha_cabecalho = i
            break

    metadados = {}

    if linha_cabecalho is not None:

        for linha in linhas[:linha_cabecalho]:

            partes = linha.split(
                ";",
                1
            )

            if len(partes) == 2:

                chave = normalizar_texto(
                    partes[0]
                )

                valor = partes[1].strip()

                metadados[chave] = valor

    return {
        "linha_cabecalho": linha_cabecalho,
        "metadados": metadados,
        "linhas_iniciais": linhas
    }

In [83]:
with zipfile.ZipFile(ZIP_2019) as z:

    diagnostico_a001 = inspecionar_arquivo_inmet(
        z,
        nome_a001
    )

print("Linha do cabeçalho:")
print(
    diagnostico_a001["linha_cabecalho"]
)

print("\nMetadados:")

for chave, valor in diagnostico_a001["metadados"].items():
    print(
        chave,
        "->",
        valor
    )

Linha do cabeçalho:
8

Metadados:
regi_o -> CO
uf -> DF
estac_o -> BRASILIA
codigo_wmo -> A001
latitude -> -15,789343
longitude -> -47,925756
altitude -> 1160,96
data_de_fundac_o -> 07/05/00


In [84]:
linha_header_a001 = (
    diagnostico_a001[
        "linha_cabecalho"
    ]
)

with zipfile.ZipFile(ZIP_2019) as z:

    with z.open(nome_a001) as arquivo:

        a001_2019_raw = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            skiprows=linha_header_a001,
            dtype=str
        )

print("Dimensão RAW:")
print(a001_2019_raw.shape)

print("\nColunas:")

for coluna in a001_2019_raw.columns:
    print(coluna)

Dimensão RAW:
(8760, 20)

Colunas:
Data
Hora UTC
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)
RADIACAO GLOBAL (KJ/m²)
TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)
TEMPERATURA DO PONTO DE ORVALHO (°C)
TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)
TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)
TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)
TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)
UMIDADE RELATIVA DO AR, HORARIA (%)
VENTO, DIREÇÃO HORARIA (gr) (° (gr))
VENTO, RAJADA MAXIMA (m/s)
VENTO, VELOCIDADE HORARIA (m/s)
Unnamed: 19


In [85]:
colunas_precipitacao = [
    coluna
    for coluna in a001_2019_raw.columns
    if "precipitacao" in normalizar_texto(coluna)
]

print("Colunas de precipitação:")
print(colunas_precipitacao)

Colunas de precipitação:
['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)']


In [86]:
colunas_precipitacao = [
    coluna
    for coluna in a001_2019_raw.columns
    if "precipitacao" in normalizar_texto(coluna)
]

print("Colunas de precipitação:")
print(colunas_precipitacao)

Colunas de precipitação:
['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)']


In [87]:
COL_PRECIP_RAW = colunas_precipitacao[0]

precipitacao_a001_raw = (
    a001_2019_raw[
        COL_PRECIP_RAW
    ]
    .astype("string")
    .str.strip()
    .str.replace(
        ",",
        ".",
        regex=False
    )
)

precipitacao_a001_raw = pd.to_numeric(
    precipitacao_a001_raw,
    errors="coerce"
)

# códigos de ausência do INMET
precipitacao_a001_raw = precipitacao_a001_raw.mask(
    precipitacao_a001_raw <= -9990
)

In [88]:
print("A001 / Brasília / 2019 - RAW ORIGINAL")

print("\nRegistros:")
print(len(precipitacao_a001_raw))

print("\nValores válidos:")
print(
    precipitacao_a001_raw
    .notna()
    .sum()
)

print("\nAusentes:")
print(
    precipitacao_a001_raw
    .isna()
    .sum()
)

print("\nValores = 0:")
print(
    (
        precipitacao_a001_raw == 0
    ).sum()
)

print("\nValores > 0:")
print(
    (
        precipitacao_a001_raw > 0
    ).sum()
)

print("\nMaior precipitação horária:")
print(
    precipitacao_a001_raw.max()
)

print("\nPrecipitação acumulada:")
print(
    precipitacao_a001_raw.sum(
        min_count=1
    )
)

A001 / Brasília / 2019 - RAW ORIGINAL

Registros:
8760

Valores válidos:
8744

Ausentes:
16

Valores = 0:
8259

Valores > 0:
485

Maior precipitação horária:
43.4

Precipitação acumulada:
1369.4


In [89]:
def diagnosticar_precipitacao(
    zip_aberto,
    nome_arquivo
):

    info = inspecionar_arquivo_inmet(
        zip_aberto,
        nome_arquivo
    )

    linha_header = info[
        "linha_cabecalho"
    ]

    meta = info[
        "metadados"
    ]

    if linha_header is None:
        raise ValueError(
            "Cabeçalho não identificado."
        )

    with zip_aberto.open(
        nome_arquivo
    ) as arquivo:

        df = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            skiprows=linha_header,
            dtype=str,
            usecols=lambda coluna:
                "precipitacao"
                in normalizar_texto(coluna)
        )

    if df.shape[1] == 0:
        raise ValueError(
            "Coluna de precipitação não encontrada."
        )

    coluna = df.columns[0]

    precipitacao = (
        df[coluna]
        .astype("string")
        .str.strip()
        .str.replace(
            ",",
            ".",
            regex=False
        )
    )

    precipitacao = pd.to_numeric(
        precipitacao,
        errors="coerce"
    )

    precipitacao = precipitacao.mask(
        precipitacao <= -9990
    )

    return {
        "uf": meta.get("uf"),
        "regiao": meta.get("regiao"),
        "codigo_wmo": (
            meta.get("codigo_wmo")
            or meta.get("codigo_wmo_")
        ),
        "estacao": meta.get("estacao"),
        "latitude": meta.get("latitude"),
        "longitude": meta.get("longitude"),
        "registros": len(precipitacao),
        "validos": int(
            precipitacao.notna().sum()
        ),
        "ausentes": int(
            precipitacao.isna().sum()
        ),
        "zeros": int(
            (precipitacao == 0).sum()
        ),
        "positivos": int(
            (precipitacao > 0).sum()
        ),
        "precipitacao_max_mm": (
            precipitacao.max()
        ),
        "precipitacao_acumulada_mm": (
            precipitacao.sum(
                min_count=1
            )
        )
    }

In [90]:
UFS_PROJETO = {
    "DF",
    "GO",
    "MT",
    "MS",
    "PR",
    "RS",
    "SC"
}

diagnosticos_raw = []

erros_raw = []

for ano in ANOS_PROJETO:

    caminho_zip = zips_por_ano[
        ano
    ]

    print(
        "\nProcessando",
        ano,
        "..."
    )

    with zipfile.ZipFile(
        caminho_zip
    ) as z:

        csvs = [
            nome
            for nome in z.namelist()
            if nome.lower().endswith(
                ".csv"
            )
        ]

        for nome in csvs:

            try:

                info = inspecionar_arquivo_inmet(
                    z,
                    nome
                )

                uf = info[
                    "metadados"
                ].get("uf")

                # ignora regiões fora
                # do nosso projeto
                if uf not in UFS_PROJETO:
                    continue

                resultado = (
                    diagnosticar_precipitacao(
                        z,
                        nome
                    )
                )

                resultado[
                    "ano"
                ] = ano

                resultado[
                    "arquivo"
                ] = nome

                diagnosticos_raw.append(
                    resultado
                )

            except Exception as erro:

                erros_raw.append(
                    {
                        "ano": ano,
                        "arquivo": nome,
                        "erro": str(erro)
                    }
                )

    print(
        "Concluído:",
        ano
    )


Processando 2019 ...
Concluído: 2019

Processando 2020 ...
Concluído: 2020

Processando 2021 ...
Concluído: 2021

Processando 2022 ...
Concluído: 2022

Processando 2023 ...
Concluído: 2023

Processando 2024 ...
Concluído: 2024


In [91]:
diagnostico_raw = pd.DataFrame(
    diagnosticos_raw
)

erros_raw_df = pd.DataFrame(
    erros_raw
)

print("Arquivos analisados:")
print(len(diagnostico_raw))

print("\nErros:")
print(len(erros_raw_df))

Arquivos analisados:
1195

Erros:
0


In [92]:
display(
    diagnostico_raw[
        [
            "ano",
            "uf",
            "codigo_wmo",
            "estacao",
            "registros",
            "validos",
            "ausentes",
            "zeros",
            "positivos",
            "precipitacao_max_mm",
            "precipitacao_acumulada_mm"
        ]
    ]
    .sort_values(
        [
            "ano",
            "uf",
            "codigo_wmo"
        ]
    )
)

,ano,uf,codigo_wmo,estacao,registros,validos,ausentes,zeros,positivos,precipitacao_max_mm,precipitacao_acumulada_mm
0,2019,DF,A001,None,8760,8744,16,8259,485,43.4,1369.4
1,2019,DF,A042,None,8760,8463,297,8044,419,59.6,1466.2
2,2019,DF,A045,None,8760,8710,50,8266,444,56.0,1369.6
3,2019,DF,A046,None,8760,8759,1,8242,517,43.4,1247.8
4,2019,DF,A047,None,8760,8723,37,8269,454,47.0,1187.4
...,...,...,...,...,...,...,...,...,...,...,...
1190,2024,SC,A867,ARARANGUA,8784,8784,0,7624,1160,34.2,1865.6
1191,2024,SC,A868,ITAJAI,8784,8700,84,7511,1189,32.2,2104.4
1192,2024,SC,A870,RANCHO QUEIMADO,8784,6499,2285,5244,1255,23.2,1619.0
1193,2024,SC,A895,CHAPECO,8784,8609,175,7813,796,28.8,2097.8


In [93]:
diagnostico_raw[
    "possui_chuva"
] = (
    diagnostico_raw[
        "positivos"
    ] > 0
)

In [94]:
resumo_raw_uf_ano = (
    diagnostico_raw
    .groupby(
        [
            "ano",
            "uf"
        ],
        as_index=False
    )
    .agg(
        arquivos=(
            "arquivo",
            "count"
        ),
        estacoes=(
            "codigo_wmo",
            "nunique"
        ),
        arquivos_com_chuva=(
            "possui_chuva",
            "sum"
        ),
        eventos_chuva=(
            "positivos",
            "sum"
        ),
        maior_precipitacao_mm=(
            "precipitacao_max_mm",
            "max"
        )
    )
)

resumo_raw_uf_ano[
    "pct_arquivos_com_chuva"
] = (
    resumo_raw_uf_ano[
        "arquivos_com_chuva"
    ]
    / resumo_raw_uf_ano[
        "arquivos"
    ]
    * 100
)

display(
    resumo_raw_uf_ano
)

,ano,uf,arquivos,estacoes,arquivos_com_chuva,eventos_chuva,maior_precipitacao_mm,pct_arquivos_com_chuva
0,2019,DF,5,5,5,2319,59.6,100.000000
1,2019,GO,26,26,26,11603,77.2,100.000000
2,2019,MS,45,45,44,17120,67.8,97.777778
3,2019,MT,39,39,39,16788,71.0,100.000000
4,2019,PR,26,26,26,15379,67.6,100.000000
5,2019,RS,44,44,44,32367,57.6,100.000000
6,2019,SC,24,24,24,19946,73.0,100.000000
7,2020,DF,5,5,5,3443,96.0,100.000000
8,2020,GO,26,26,26,11793,81.6,100.000000
9,2020,MS,45,45,36,10155,73.8,80.000000


In [95]:
suspeitos_raw = (
    diagnostico_raw[
        (
            diagnostico_raw[
                "validos"
            ] >= 5000
        )
        &
        (
            diagnostico_raw[
                "positivos"
            ] == 0
        )
    ]
    [
        [
            "ano",
            "uf",
            "codigo_wmo",
            "estacao",
            "validos",
            "zeros",
            "positivos",
            "precipitacao_max_mm"
        ]
    ]
    .sort_values(
        [
            "ano",
            "uf",
            "codigo_wmo"
        ]
    )
)

print(
    "Estação-ano com >=5000 registros válidos e nenhuma chuva:"
)

print(
    len(suspeitos_raw)
)

display(
    suspeitos_raw
)

Estação-ano com >=5000 registros válidos e nenhuma chuva:
0


,ano,uf,codigo_wmo,estacao,validos,zeros,positivos,precipitacao_max_mm


In [96]:
resumo_ano = (
    diagnostico_raw
    .groupby(
        "ano",
        as_index=False
    )
    .agg(
        arquivos=(
            "arquivo",
            "count"
        ),
        estacoes=(
            "codigo_wmo",
            "nunique"
        ),
        registros_validos=(
            "validos",
            "sum"
        ),
        eventos_chuva=(
            "positivos",
            "sum"
        ),
        arquivos_com_chuva=(
            "possui_chuva",
            "sum"
        ),
        maior_precipitacao_mm=(
            "precipitacao_max_mm",
            "max"
        )
    )
)

resumo_ano[
    "pct_arquivos_com_chuva"
] = (
    resumo_ano[
        "arquivos_com_chuva"
    ]
    / resumo_ano[
        "arquivos"
    ]
    * 100
)

display(
    resumo_ano
)

,ano,arquivos,estacoes,registros_validos,eventos_chuva,arquivos_com_chuva,maior_precipitacao_mm,pct_arquivos_com_chuva
0,2019,209,209,1551201,115522,208,77.2,99.521531
1,2020,208,208,1357552,91456,197,96.0,94.711538
2,2021,207,207,951841,72498,161,74.2,77.777778
3,2022,190,190,1095023,85934,185,75.4,97.368421
4,2023,191,191,1392075,113705,188,87.0,98.429319
5,2024,190,190,1307975,107940,178,85.0,93.684211


In [97]:
suspeitos_raw = (
    diagnostico_raw[
        (diagnostico_raw["validos"] >= 5000)
        &
        (diagnostico_raw["positivos"] == 0)
    ]
    [
        [
            "ano",
            "uf",
            "codigo_wmo",
            "estacao",
            "validos",
            "zeros",
            "positivos",
            "precipitacao_max_mm",
            "precipitacao_acumulada_mm"
        ]
    ]
    .sort_values(
        [
            "ano",
            "uf",
            "codigo_wmo"
        ]
    )
)

print(
    "Estações-ano com >= 5000 registros válidos "
    "e nenhuma precipitação positiva:"
)

print(len(suspeitos_raw))

display(suspeitos_raw)

Estações-ano com >= 5000 registros válidos e nenhuma precipitação positiva:
0


,ano,uf,codigo_wmo,estacao,validos,zeros,positivos,precipitacao_max_mm,precipitacao_acumulada_mm


In [98]:
AUDIT_DIR = (
    BASE_DIR
    / "data"
    / "processed"
    / "inmet_auditoria"
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(AUDIT_DIR)
print(AUDIT_DIR.exists())

C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\processed\inmet_auditoria
True


In [99]:
diagnostico_raw.to_csv(
    AUDIT_DIR
    / "diagnostico_precipitacao_inmet_raw_2019_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

resumo_raw_uf_ano.to_csv(
    AUDIT_DIR
    / "resumo_precipitacao_uf_ano.csv",
    index=False,
    encoding="utf-8-sig"
)

resumo_ano.to_csv(
    AUDIT_DIR
    / "resumo_precipitacao_ano.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos de auditoria salvos.")

Arquivos de auditoria salvos.


In [100]:
suspeitos_raw.to_csv(
    AUDIT_DIR
    / "estacoes_precipitacao_suspeita.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Auditoria dos casos suspeitos salva.")

Auditoria dos casos suspeitos salva.


# Reconstrução definitiva da base INMET

A partir desta seção, a pipeline climática passa a utilizar exclusivamente
os arquivos ZIP originais disponibilizados pelo INMET.

A base anteriormente processada (`INMET_LIMPO`) foi descontinuada para uso
analítico após a auditoria identificar perda de valores de precipitação.

Os arquivos brutos originais não serão modificados nem extraídos manualmente.
A leitura será realizada diretamente dos arquivos ZIP.

Período analisado: 2019–2024.

Regiões consideradas nesta etapa:
- Centro-Oeste
- Sul

UFs:
DF, GO, MS, MT, PR, RS e SC.

Nova arquitetura:

RAW → PROCESSED → CURATED

In [101]:
from pathlib import Path

ANOS_PROJETO = list(range(2019, 2025))

UFS_PROJETO = {
    "DF",
    "GO",
    "MS",
    "MT",
    "PR",
    "RS",
    "SC"
}

RAW_INMET_DIR = (
    BASE_DIR
    / "data"
    / "raw"
    / "inmet"
)

PROCESSED_DIR = (
    BASE_DIR
    / "data"
    / "databases_processed"
    / "inmet"
)

CURATED_DIR = (
    BASE_DIR
    / "data"
    / "databases_curated"
    / "inmet"
)

RELATORIOS_DIR = (
    BASE_DIR
    / "relatorios"
    / "inmet"
)

PROCESSED_HOURLY_DIR = PROCESSED_DIR / "hourly"
PROCESSED_DAILY_DIR = PROCESSED_DIR / "daily"
PROCESSED_STATION_YEAR_DIR = PROCESSED_DIR / "station_year"
PROCESSED_QUALITY_DIR = PROCESSED_DIR / "quality"

CURATED_MUNICIPIO_ANO_DIR = CURATED_DIR / "municipio_ano"
CURATED_POWERBI_DIR = CURATED_DIR / "powerbi"

pastas = [
    PROCESSED_HOURLY_DIR,
    PROCESSED_DAILY_DIR,
    PROCESSED_STATION_YEAR_DIR,
    PROCESSED_QUALITY_DIR,
    CURATED_MUNICIPIO_ANO_DIR,
    CURATED_POWERBI_DIR,
    RELATORIOS_DIR
]

for pasta in pastas:
    pasta.mkdir(
        parents=True,
        exist_ok=True
    )

print("Estrutura criada:")
print()

for pasta in pastas:
    print(
        pasta.relative_to(BASE_DIR),
        "->",
        pasta.exists()
    )

Estrutura criada:

data\databases_processed\inmet\hourly -> True
data\databases_processed\inmet\daily -> True
data\databases_processed\inmet\station_year -> True
data\databases_processed\inmet\quality -> True
data\databases_curated\inmet\municipio_ano -> True
data\databases_curated\inmet\powerbi -> True
relatorios\inmet -> True


In [102]:
RELATORIO_REINICIO = (
    RELATORIOS_DIR
    / "relatorio_reinicio_pipeline_inmet_2019_2024.md"
)

conteudo_relatorio = """# Relatório de Reinício da Pipeline INMET

## Projeto AgroESG Analytics

### Período

2019 a 2024.

### Fonte

Instituto Nacional de Meteorologia - INMET.

Os dados utilizados a partir desta etapa são os arquivos ZIP originais
baixados diretamente da fonte oficial.

---

## Problema identificado

Durante a validação da base previamente processada (`INMET_LIMPO`),
foram encontrados valores incompatíveis para precipitação.

Um exemplo importante foi a estação A001 - Brasília.

Na base anteriormente processada, a precipitação anual de 2019 aparecia
como 0 mm.

Ao consultar diretamente o arquivo original do INMET foram encontrados:

- 8760 registros horários;
- 8744 registros válidos de precipitação;
- 16 registros ausentes;
- 8259 registros com precipitação igual a zero;
- 485 registros com precipitação maior que zero;
- precipitação horária máxima de 43,4 mm;
- precipitação acumulada de 1369,4 mm.

Isso demonstrou que os valores de precipitação existiam na fonte bruta
e foram perdidos ou transformados incorretamente durante o processamento
anterior.

---

## Auditoria geral

Foram analisados 1195 arquivos estação-ano correspondentes ao período
2019-2024 para as UFs DF, GO, MS, MT, PR, RS e SC.

Resultado da leitura:

- arquivos analisados: 1195;
- erros de leitura: 0.

Também foi executada uma busca por arquivos contendo pelo menos
5000 registros válidos de precipitação e nenhuma precipitação positiva.

Resultado:

- casos encontrados: 0.

---

## Decisão metodológica

A base `INMET_LIMPO` deixa de ser utilizada na construção dos indicadores
climáticos finais.

A nova pipeline utilizará exclusivamente os arquivos ZIP brutos originais
do INMET.

Os arquivos brutos permanecerão imutáveis.

A leitura será realizada diretamente dos ZIPs, sem necessidade de
descompactação manual.

---

## Nova arquitetura

RAW
→ PROCESSED HOURLY
→ PROCESSED DAILY
→ PROCESSED STATION YEAR
→ associação espacial estação-município
→ CURATED MUNICIPIO ANO
→ integração com as demais bases do projeto
→ Power BI

---

## Status

Auditoria da fonte bruta concluída.

Fonte RAW aprovada para reconstrução da pipeline climática.
"""

RELATORIO_REINICIO.write_text(
    conteudo_relatorio,
    encoding="utf-8"
)

print("Relatório criado:")
print(
    RELATORIO_REINICIO.relative_to(BASE_DIR)
)

print()
print("Existe?")
print(RELATORIO_REINICIO.exists())

Relatório criado:
relatorios\inmet\relatorio_reinicio_pipeline_inmet_2019_2024.md

Existe?
True


In [103]:
diagnostico_raw.to_csv(
    PROCESSED_QUALITY_DIR
    / "diagnostico_precipitacao_raw_2019_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

resumo_raw_uf_ano.to_csv(
    PROCESSED_QUALITY_DIR
    / "resumo_precipitacao_uf_ano.csv",
    index=False,
    encoding="utf-8-sig"
)

resumo_ano.to_csv(
    PROCESSED_QUALITY_DIR
    / "resumo_precipitacao_ano.csv",
    index=False,
    encoding="utf-8-sig"
)

suspeitos_raw.to_csv(
    PROCESSED_QUALITY_DIR
    / "estacoes_precipitacao_suspeita.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos de qualidade salvos:")
print()

for arquivo in sorted(PROCESSED_QUALITY_DIR.glob("*.csv")):
    print(arquivo.name)

Arquivos de qualidade salvos:

diagnostico_precipitacao_raw_2019_2024.csv
estacoes_precipitacao_suspeita.csv
resumo_precipitacao_ano.csv
resumo_precipitacao_uf_ano.csv


In [104]:
import re
import unicodedata


def normalizar_texto(texto):

    if texto is None:
        return None

    texto = str(texto).strip()

    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = "".join(
        caractere
        for caractere in texto
        if not unicodedata.combining(caractere)
    )

    texto = texto.lower()

    texto = re.sub(
        r"[^a-z0-9]+",
        "_",
        texto
    )

    texto = texto.strip("_")

    return texto

In [105]:
testes = [
    "ESTAÇÃO:",
    "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)",
    "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",
    "UMIDADE RELATIVA DO AR, HORARIA (%)",
    "Código (WMO)"
]

for teste in testes:
    print(
        teste,
        "->",
        normalizar_texto(teste)
    )

ESTAÇÃO: -> estacao
PRECIPITAÇÃO TOTAL, HORÁRIO (mm) -> precipitacao_total_horario_mm
TEMPERATURA DO AR - BULBO SECO, HORARIA (°C) -> temperatura_do_ar_bulbo_seco_horaria_c
UMIDADE RELATIVA DO AR, HORARIA (%) -> umidade_relativa_do_ar_horaria
Código (WMO) -> codigo_wmo


In [106]:
print("RAW INMET:")
print(RAW_INMET_DIR)

print("\nExiste?")
print(RAW_INMET_DIR.exists())

print("\nZIPs encontrados:")

for ano in ANOS_PROJETO:

    pasta_ano = RAW_INMET_DIR / str(ano)

    arquivos_zip = sorted(
        pasta_ano.glob("*.zip")
    )

    print(
        ano,
        "->",
        len(arquivos_zip),
        "ZIP(s)",
        [arquivo.name for arquivo in arquivos_zip]
    )

RAW INMET:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\raw\inmet

Existe?
True

ZIPs encontrados:
2019 -> 1 ZIP(s) ['arquivo_inmet_2019.zip']
2020 -> 1 ZIP(s) ['arquivo_inmet_2020.zip']
2021 -> 1 ZIP(s) ['arquivo_inmet_2021.zip']
2022 -> 1 ZIP(s) ['arquivo_inmet_2022.zip']
2023 -> 1 ZIP(s) ['arquivo_inmet_2023.zip']
2024 -> 1 ZIP(s) ['arquivo_inmet_2024.zip']


In [107]:
print("RAW INMET:")
print(RAW_INMET_DIR)

print("\nExiste?")
print(RAW_INMET_DIR.exists())

print("\nZIPs encontrados:")

for ano in ANOS_PROJETO:

    pasta_ano = RAW_INMET_DIR / str(ano)

    arquivos_zip = sorted(
        pasta_ano.glob("*.zip")
    )

    print(
        ano,
        "->",
        len(arquivos_zip),
        "ZIP(s)",
        [arquivo.name for arquivo in arquivos_zip]
    )

RAW INMET:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\raw\inmet

Existe?
True

ZIPs encontrados:
2019 -> 1 ZIP(s) ['arquivo_inmet_2019.zip']
2020 -> 1 ZIP(s) ['arquivo_inmet_2020.zip']
2021 -> 1 ZIP(s) ['arquivo_inmet_2021.zip']
2022 -> 1 ZIP(s) ['arquivo_inmet_2022.zip']
2023 -> 1 ZIP(s) ['arquivo_inmet_2023.zip']
2024 -> 1 ZIP(s) ['arquivo_inmet_2024.zip']


In [109]:
import zipfile
import pandas as pd

manifesto = []

for ano in ANOS_PROJETO:

    pasta_ano = RAW_INMET_DIR / str(ano)

    arquivos_zip = sorted(
        pasta_ano.glob("*.zip")
    )

    for caminho_zip in arquivos_zip:

        with zipfile.ZipFile(caminho_zip) as z:

            nomes = z.namelist()

            csvs = [
                nome
                for nome in nomes
                if nome.lower().endswith(".csv")
            ]

            manifesto.append(
                {
                    "ano": ano,
                    "arquivo_zip": caminho_zip.name,
                    "tamanho_mb": round(
                        caminho_zip.stat().st_size / (1024 ** 2),
                        2
                    ),
                    "itens_zip": len(nomes),
                    "arquivos_csv": len(csvs)
                }
            )

manifesto_inmet = pd.DataFrame(manifesto)

display(manifesto_inmet)

manifesto_inmet.to_csv(
    PROCESSED_QUALITY_DIR
    / "manifesto_fontes_inmet_2019_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nManifesto salvo em:")
print(
    PROCESSED_QUALITY_DIR
    / "manifesto_fontes_inmet_2019_2024.csv"
)

,ano,arquivo_zip,tamanho_mb,itens_zip,arquivos_csv
0,2019,arquivo_inmet_2019.zip,112.11,590,589
1,2020,arquivo_inmet_2020.zip,98.85,589,589
2,2021,arquivo_inmet_2021.zip,76.84,588,588
3,2022,arquivo_inmet_2022.zip,86.18,567,567
4,2023,arquivo_inmet_2023.zip,102.11,567,567
5,2024,arquivo_inmet_2024.zip,98.01,565,565



Manifesto salvo em:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_processed\inmet\quality\manifesto_fontes_inmet_2019_2024.csv


In [110]:
print(manifesto_inmet.shape)
print(manifesto_inmet.columns.tolist())

(6, 5)
['ano', 'arquivo_zip', 'tamanho_mb', 'itens_zip', 'arquivos_csv']


In [111]:
integridade_zips = []

for ano in ANOS_PROJETO:

    caminho_zip = zips_por_ano[ano]

    with zipfile.ZipFile(caminho_zip) as z:

        arquivo_corrompido = z.testzip()

        integridade_zips.append(
            {
                "ano": ano,
                "arquivo_zip": caminho_zip.name,
                "zip_integro": arquivo_corrompido is None,
                "arquivo_corrompido": arquivo_corrompido
            }
        )

integridade_zips = pd.DataFrame(integridade_zips)

display(integridade_zips)

,ano,arquivo_zip,zip_integro,arquivo_corrompido
0,2019,arquivo_inmet_2019.zip,True,None
1,2020,arquivo_inmet_2020.zip,True,None
2,2021,arquivo_inmet_2021.zip,True,None
3,2022,arquivo_inmet_2022.zip,True,None
4,2023,arquivo_inmet_2023.zip,True,None
5,2024,arquivo_inmet_2024.zip,True,None


In [112]:
integridade_zips.to_csv(
    PROCESSED_QUALITY_DIR
    / "integridade_zips_inmet_2019_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Auditoria de integridade salva.")

Auditoria de integridade salva.


In [113]:
integridade_zips = []

for ano in ANOS_PROJETO:

    caminho_zip = zips_por_ano[ano]

    with zipfile.ZipFile(caminho_zip) as z:

        arquivo_corrompido = z.testzip()

        integridade_zips.append(
            {
                "ano": ano,
                "arquivo_zip": caminho_zip.name,
                "zip_integro": arquivo_corrompido is None,
                "arquivo_corrompido": arquivo_corrompido
            }
        )

integridade_zips = pd.DataFrame(integridade_zips)

display(integridade_zips)

,ano,arquivo_zip,zip_integro,arquivo_corrompido
0,2019,arquivo_inmet_2019.zip,True,None
1,2020,arquivo_inmet_2020.zip,True,None
2,2021,arquivo_inmet_2021.zip,True,None
3,2022,arquivo_inmet_2022.zip,True,None
4,2023,arquivo_inmet_2023.zip,True,None
5,2024,arquivo_inmet_2024.zip,True,None


In [114]:
integridade_zips.to_csv(
    PROCESSED_QUALITY_DIR
    / "integridade_zips_inmet_2019_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Auditoria de integridade salva.")

Auditoria de integridade salva.


# ETL definitivo do INMET a partir dos arquivos RAW

Após a auditoria dos arquivos ZIP originais do INMET, inicia-se nesta seção
o processamento definitivo dos dados climáticos.

Os arquivos da camada RAW permanecem imutáveis.

Fluxo:

RAW ZIP
→ PROCESSED hourly
→ PROCESSED daily
→ PROCESSED station_year
→ integração espacial com Malha Municipal IBGE 2024
→ CURATED municipio_ano

In [115]:
from pathlib import Path
import re
import unicodedata
import zipfile

import numpy as np
import pandas as pd


ANOS_PROJETO = list(range(2019, 2025))


RAW_INMET_DIR = (
    BASE_DIR
    / "data"
    / "raw"
    / "inmet"
)


PROCESSED_INMET_DIR = (
    BASE_DIR
    / "data"
    / "databases_processed"
    / "inmet"
)


PROCESSED_HOURLY_DIR = (
    PROCESSED_INMET_DIR
    / "hourly"
)


PROCESSED_DAILY_DIR = (
    PROCESSED_INMET_DIR
    / "daily"
)


PROCESSED_STATION_YEAR_DIR = (
    PROCESSED_INMET_DIR
    / "station_year"
)


PROCESSED_QUALITY_DIR = (
    PROCESSED_INMET_DIR
    / "quality"
)


CURATED_INMET_DIR = (
    BASE_DIR
    / "data"
    / "databases_curated"
    / "inmet"
)


CURATED_MUNICIPIO_ANO_DIR = (
    CURATED_INMET_DIR
    / "municipio_ano"
)


for pasta in [
    PROCESSED_HOURLY_DIR,
    PROCESSED_DAILY_DIR,
    PROCESSED_STATION_YEAR_DIR,
    PROCESSED_QUALITY_DIR,
    CURATED_MUNICIPIO_ANO_DIR
]:
    pasta.mkdir(
        parents=True,
        exist_ok=True
    )


print("Estrutura criada com sucesso.")

Estrutura criada com sucesso.


In [116]:
zips_por_ano = {}


for ano in ANOS_PROJETO:

    pasta_ano = (
        RAW_INMET_DIR
        / str(ano)
    )

    arquivos_zip = sorted(
        pasta_ano.glob("*.zip")
    )

    if len(arquivos_zip) != 1:

        raise ValueError(
            f"{ano}: esperado 1 ZIP, encontrados {len(arquivos_zip)}"
        )

    zips_por_ano[ano] = arquivos_zip[0]


for ano, caminho in zips_por_ano.items():

    print(
        ano,
        "->",
        caminho.name
    )

2019 -> arquivo_inmet_2019.zip
2020 -> arquivo_inmet_2020.zip
2021 -> arquivo_inmet_2021.zip
2022 -> arquivo_inmet_2022.zip
2023 -> arquivo_inmet_2023.zip
2024 -> arquivo_inmet_2024.zip


In [117]:
def normalizar_texto(texto):

    texto = str(texto)

    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = "".join(
        caractere
        for caractere in texto
        if not unicodedata.combining(caractere)
    )

    texto = texto.lower()

    texto = re.sub(
        r"[^a-z0-9]+",
        "_",
        texto
    )

    return texto.strip("_")

In [118]:
def localizar_coluna(
    dataframe,
    inclui,
    exclui=()
):

    for coluna in dataframe.columns:

        coluna_normalizada = normalizar_texto(
            coluna
        )

        contem_incluidos = all(
            termo in coluna_normalizada
            for termo in inclui
        )

        contem_excluidos = any(
            termo in coluna_normalizada
            for termo in exclui
        )

        if (
            contem_incluidos
            and not contem_excluidos
        ):
            return coluna

    return None

In [119]:
def converter_numerico(serie):

    serie = (
        serie
        .astype("string")
        .str.strip()
        .str.replace(
            ",",
            ".",
            regex=False
        )
    )

    serie = pd.to_numeric(
        serie,
        errors="coerce"
    )

    serie = serie.replace(
        [-9999, -9999.0],
        np.nan
    )

    return serie

In [120]:
def inspecionar_membro_inmet(
    zip_aberto,
    nome_arquivo,
    max_linhas=25
):

    linhas = []

    with zip_aberto.open(
        nome_arquivo
    ) as arquivo:

        for _ in range(max_linhas):

            linha = arquivo.readline()

            if not linha:
                break

            linhas.append(
                linha.decode(
                    "latin1",
                    errors="replace"
                ).strip()
            )


    linha_cabecalho = None


    for indice, linha in enumerate(linhas):

        linha_norm = normalizar_texto(
            linha
        )

        if (
            linha_norm.startswith("data_")
            and "precipitacao" in linha_norm
        ):

            linha_cabecalho = indice
            break


    if linha_cabecalho is None:

        raise ValueError(
            f"Cabeçalho não identificado: {nome_arquivo}"
        )


    metadados = {}


    for linha in linhas[:linha_cabecalho]:

        partes = linha.split(
            ";",
            1
        )

        if len(partes) == 2:

            chave = normalizar_texto(
                partes[0]
            )

            valor = partes[1].strip()

            metadados[chave] = valor


    return {
        "linha_cabecalho": linha_cabecalho,
        "metadados": metadados
    }

In [122]:
teste_parser = []


for ano in ANOS_PROJETO:

    caminho_zip = zips_por_ano[ano]

    with zipfile.ZipFile(
        caminho_zip
    ) as z:

        csvs = [
            nome
            for nome in z.namelist()
            if nome.lower().endswith(".csv")
        ]

        nome_teste = csvs[0]

        df_teste, mapa_teste = ler_membro_inmet(
            z,
            nome_teste,
            ano
        )


        teste_parser.append(
            {
                "ano": ano,
                "arquivo": nome_teste,
                "uf": df_teste["uf"].iloc[0],
                "codigo_wmo": df_teste["codigo_wmo"].iloc[0],
                "estacao": df_teste["estacao"].iloc[0],
                "registros": len(df_teste),
                "datas_invalidas": df_teste["data"].isna().sum(),
                "precipitacao_valida": df_teste["precipitacao_mm"].notna().sum(),
                "eventos_chuva": (df_teste["precipitacao_mm"] > 0).sum(),
                "precipitacao_max_mm": df_teste["precipitacao_mm"].max(),
                "precipitacao_total_mm": df_teste[
                    "precipitacao_mm"
                ].sum(min_count=1)
            }
        )


teste_parser = pd.DataFrame(
    teste_parser
)


display(teste_parser)

,ano,arquivo,uf,codigo_wmo,estacao,registros,datas_invalidas,precipitacao_valida,eventos_chuva,precipitacao_max_mm,precipitacao_total_mm
0,2019,2019/INMET_CO_DF_A001_BRASILIA_01-01-2019_A_31...,DF,A001,,3456,0,3454,219,43.4,643.8
1,2020,INMET_CO_DF_A001_BRASILIA_01-01-2020_A_31-12-2...,DF,A001,BRASILIA,3456,0,3456,264,30.6,596.6
2,2021,INMET_CO_DF_A001_BRASILIA_01-01-2021_A_31-12-2...,DF,A001,BRASILIA,3456,0,3455,213,54.2,897.4
3,2022,INMET_CO_DF_A001_BRASILIA_01-01-2022_A_31-12-2...,DF,A001,BRASILIA,3456,0,3455,217,36.6,608.2
4,2023,INMET_CO_DF_A001_BRASILIA_01-01-2023_A_31-12-2...,DF,A001,BRASILIA,3456,0,3455,166,23.4,362.4
5,2024,INMET_CO_DF_A001_BRASILIA_01-01-2024_A_31-12-2...,DF,A001,BRASILIA,3456,0,3448,247,35.2,572.2


In [123]:
def converter_data_inmet(serie):

    texto = (
        serie
        .astype("string")
        .str.strip()
        .str.replace(
            "/",
            "-",
            regex=False
        )
    )

    datas = pd.to_datetime(
        texto,
        format="%Y-%m-%d",
        errors="coerce"
    )

    return datas

In [125]:
def buscar_metadado(
    metadados,
    *possiveis_chaves
):

    for chave in possiveis_chaves:

        if chave in metadados:

            valor = metadados[chave]

            if pd.notna(valor):

                return str(valor).strip()

    return ""

In [149]:
# ============================================================
# MAPEAMENTO DE REGIÕES
# ============================================================

MAPA_REGIOES = {
    "N": "norte",
    "NE": "nordeste",
    "CO": "centro_oeste",
    "SE": "sudeste",
    "S": "sul"
}


# Proteção adicional:
# caso o metadado REGIAO do INMET não seja identificado,
# inferimos a região a partir da UF.

REGIAO_POR_UF = {
    "AC": "norte",
    "AL": "nordeste",
    "AP": "norte",
    "AM": "norte",
    "BA": "nordeste",
    "CE": "nordeste",
    "DF": "centro_oeste",
    "ES": "sudeste",
    "GO": "centro_oeste",
    "MA": "nordeste",
    "MT": "centro_oeste",
    "MS": "centro_oeste",
    "MG": "sudeste",
    "PA": "norte",
    "PB": "nordeste",
    "PR": "sul",
    "PE": "nordeste",
    "PI": "nordeste",
    "RJ": "sudeste",
    "RN": "nordeste",
    "RS": "sul",
    "RO": "norte",
    "RR": "norte",
    "SC": "sul",
    "SP": "sudeste",
    "SE": "nordeste",
    "TO": "norte"
}


# ============================================================
# CONVERSÃO DE METADADOS NUMÉRICOS
# ============================================================

def numero_metadado(valor):

    if valor is None:
        return np.nan

    valor = (
        str(valor)
        .strip()
        .replace(",", ".")
    )

    return pd.to_numeric(
        valor,
        errors="coerce"
    )


# ============================================================
# LEITURA DE UM ARQUIVO CSV DO INMET DENTRO DO ZIP
# ============================================================

def ler_membro_inmet(
    zip_aberto,
    nome_arquivo,
    ano
):

    # --------------------------------------------------------
    # INSPEÇÃO DO ARQUIVO
    # --------------------------------------------------------

    info = inspecionar_membro_inmet(
        zip_aberto,
        nome_arquivo
    )

    linha_header = info[
        "linha_cabecalho"
    ]

    meta = info[
        "metadados"
    ]


    # --------------------------------------------------------
    # LEITURA DO CSV
    # --------------------------------------------------------

    with zip_aberto.open(
        nome_arquivo
    ) as arquivo:

        raw = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            skiprows=linha_header,
            dtype=str,
            low_memory=False
        )


    # --------------------------------------------------------
    # REMOVE COLUNA VAZIA FINAL DO CSV
    # --------------------------------------------------------

    raw = raw.loc[
        :,
        ~raw.columns
        .astype(str)
        .str.startswith("Unnamed")
    ]


    # --------------------------------------------------------
    # IDENTIFICAÇÃO DAS COLUNAS
    # --------------------------------------------------------

    mapa = {

        "data":
            localizar_coluna(
                raw,
                ["data"]
            ),

        "hora_utc":
            localizar_coluna(
                raw,
                ["hora", "utc"]
            ),

        "precipitacao_mm":
            localizar_coluna(
                raw,
                [
                    "precipitacao",
                    "total",
                    "horario"
                ]
            ),

        "pressao_mb":
            localizar_coluna(
                raw,
                [
                    "pressao",
                    "nivel",
                    "estacao",
                    "horaria"
                ]
            ),

        "pressao_max_mb":
            localizar_coluna(
                raw,
                [
                    "pressao",
                    "max",
                    "hora",
                    "ant"
                ]
            ),

        "pressao_min_mb":
            localizar_coluna(
                raw,
                [
                    "pressao",
                    "min",
                    "hora",
                    "ant"
                ]
            ),

        "radiacao_kj_m2":
            localizar_coluna(
                raw,
                [
                    "radiacao",
                    "global"
                ]
            ),

        "temperatura_c":
            localizar_coluna(
                raw,
                [
                    "temperatura",
                    "ar",
                    "bulbo",
                    "seco"
                ]
            ),

        "ponto_orvalho_c":
            localizar_coluna(
                raw,
                [
                    "temperatura",
                    "ponto",
                    "orvalho"
                ],
                [
                    "max",
                    "min"
                ]
            ),

        "temperatura_max_c":
            localizar_coluna(
                raw,
                [
                    "temperatura",
                    "max",
                    "hora",
                    "ant"
                ],
                [
                    "orvalho"
                ]
            ),

        "temperatura_min_c":
            localizar_coluna(
                raw,
                [
                    "temperatura",
                    "min",
                    "hora",
                    "ant"
                ],
                [
                    "orvalho"
                ]
            ),

        "ponto_orvalho_max_c":
            localizar_coluna(
                raw,
                [
                    "temperatura",
                    "orvalho",
                    "max"
                ]
            ),

        "ponto_orvalho_min_c":
            localizar_coluna(
                raw,
                [
                    "temperatura",
                    "orvalho",
                    "min"
                ]
            ),

        "umidade_max_pct":
            localizar_coluna(
                raw,
                [
                    "umidade",
                    "max",
                    "hora",
                    "ant"
                ]
            ),

        "umidade_min_pct":
            localizar_coluna(
                raw,
                [
                    "umidade",
                    "min",
                    "hora",
                    "ant"
                ]
            ),

        "umidade_pct":
            localizar_coluna(
                raw,
                [
                    "umidade",
                    "relativa",
                    "ar",
                    "horaria"
                ]
            ),

        "direcao_vento_graus":
            localizar_coluna(
                raw,
                [
                    "vento",
                    "direcao",
                    "horaria"
                ]
            ),

        "rajada_max_ms":
            localizar_coluna(
                raw,
                [
                    "vento",
                    "rajada",
                    "max"
                ]
            ),

        "velocidade_vento_ms":
            localizar_coluna(
                raw,
                [
                    "vento",
                    "velocidade",
                    "horaria"
                ]
            )
    }


    # --------------------------------------------------------
    # VALIDAÇÕES ESSENCIAIS
    # --------------------------------------------------------

    if mapa["data"] is None:

        raise ValueError(
            f"Coluna de data ausente: {nome_arquivo}"
        )


    if mapa["precipitacao_mm"] is None:

        raise ValueError(
            f"Coluna de precipitação ausente: {nome_arquivo}"
        )


    # --------------------------------------------------------
    # CRIAÇÃO DO DATAFRAME PADRONIZADO
    # --------------------------------------------------------

    df = pd.DataFrame(
        index=raw.index
    )


    # --------------------------------------------------------
    # DATA
    # --------------------------------------------------------

    df["data"] = converter_data_inmet(
        raw[
            mapa["data"]
        ]
    )


    # --------------------------------------------------------
    # HORA UTC
    # --------------------------------------------------------

    if mapa["hora_utc"] is not None:

        df["hora_utc"] = (
            raw[
                mapa["hora_utc"]
            ]
            .astype("string")
            .str.extract(
                r"(\d{3,4})",
                expand=False
            )
            .str.zfill(4)
        )

    else:

        df["hora_utc"] = pd.NA


    # --------------------------------------------------------
    # VARIÁVEIS NUMÉRICAS
    # --------------------------------------------------------

    colunas_numericas = [
        "precipitacao_mm",
        "pressao_mb",
        "pressao_max_mb",
        "pressao_min_mb",
        "radiacao_kj_m2",
        "temperatura_c",
        "ponto_orvalho_c",
        "temperatura_max_c",
        "temperatura_min_c",
        "ponto_orvalho_max_c",
        "ponto_orvalho_min_c",
        "umidade_max_pct",
        "umidade_min_pct",
        "umidade_pct",
        "direcao_vento_graus",
        "rajada_max_ms",
        "velocidade_vento_ms"
    ]


    for coluna in colunas_numericas:

        coluna_raw = mapa[
            coluna
        ]

        if coluna_raw is None:

            df[coluna] = np.nan

        else:

            df[coluna] = converter_numerico(
                raw[
                    coluna_raw
                ]
            )


    # ========================================================
    # METADADOS DA ESTAÇÃO
    # ========================================================

    # --------------------------------------------------------
    # UF
    #
    # Usamos buscar_metadado em vez de meta.get diretamente
    # para deixar o parser mais resistente.
    # --------------------------------------------------------

    uf = str(
        buscar_metadado(
            meta,
            "uf"
        )
    ).strip().upper()


    # --------------------------------------------------------
    # REGIÃO
    #
    # Alguns arquivos foram normalizados como "regi_o".
    # Por isso procuramos diferentes versões da chave.
    # --------------------------------------------------------

    regiao_raw = str(
        buscar_metadado(
            meta,
            "regiao",
            "regi_o",
            "regiao_"
        )
    ).strip().upper()


    # Primeiro tenta usar o metadado original do INMET

    regiao = MAPA_REGIOES.get(
        regiao_raw
    )


    # Se não conseguir identificar pelo metadado,
    # usa a UF como fallback.

    if not regiao:

        regiao = REGIAO_POR_UF.get(
            uf,
            pd.NA
        )


    # --------------------------------------------------------
    # GRAVA METADADOS
    # --------------------------------------------------------

    df["ano"] = ano

    df["uf"] = uf

    df["regiao"] = regiao


    df["codigo_wmo"] = str(
        buscar_metadado(
            meta,
            "codigo_wmo",
            "codigo"
        )
    ).strip().upper()


    df["estacao"] = str(
        buscar_metadado(
            meta,
            "estacao",
            "estac_o",
            "estacao_"
        )
    ).strip()


    df["latitude"] = numero_metadado(
        meta.get(
            "latitude"
        )
    )


    df["longitude"] = numero_metadado(
        meta.get(
            "longitude"
        )
    )


    df["altitude_m"] = numero_metadado(
        meta.get(
            "altitude"
        )
    )


    df["arquivo_origem"] = (
        nome_arquivo
    )


    # --------------------------------------------------------
    # REORGANIZA ÍNDICE
    # --------------------------------------------------------

    df = df.reset_index(
        drop=True
    )


    return df, mapa

In [127]:
teste_parser = []

for ano in ANOS_PROJETO:

    caminho_zip = zips_por_ano[ano]

    with zipfile.ZipFile(
        caminho_zip
    ) as z:

        csvs = [
            nome
            for nome in z.namelist()
            if nome.lower().endswith(".csv")
        ]

        nome_teste = csvs[0]

        df_teste, mapa_teste = ler_membro_inmet(
            z,
            nome_teste,
            ano
        )

        teste_parser.append(
            {
                "ano": ano,
                "arquivo": nome_teste,
                "uf": df_teste["uf"].iloc[0],
                "codigo_wmo": df_teste["codigo_wmo"].iloc[0],
                "estacao": df_teste["estacao"].iloc[0],
                "registros": len(df_teste),
                "datas_invalidas": df_teste[
                    "data"
                ].isna().sum(),
                "data_min": df_teste[
                    "data"
                ].min(),
                "data_max": df_teste[
                    "data"
                ].max(),
                "precipitacao_valida": df_teste[
                    "precipitacao_mm"
                ].notna().sum(),
                "eventos_chuva": (
                    df_teste[
                        "precipitacao_mm"
                    ] > 0
                ).sum(),
                "precipitacao_max_mm": df_teste[
                    "precipitacao_mm"
                ].max(),
                "precipitacao_total_mm": df_teste[
                    "precipitacao_mm"
                ].sum(
                    min_count=1
                )
            }
        )


teste_parser = pd.DataFrame(
    teste_parser
)

display(teste_parser)

,ano,arquivo,uf,codigo_wmo,estacao,registros,datas_invalidas,data_min,data_max,precipitacao_valida,eventos_chuva,precipitacao_max_mm,precipitacao_total_mm
0,2019,2019/INMET_CO_DF_A001_BRASILIA_01-01-2019_A_31...,DF,A001,BRASILIA,8760,0,2019-01-01,2019-12-31,8744,485,43.4,1369.4
1,2020,INMET_CO_DF_A001_BRASILIA_01-01-2020_A_31-12-2...,DF,A001,BRASILIA,8784,0,2020-01-01,2020-12-31,8784,697,37.6,1576.6
2,2021,INMET_CO_DF_A001_BRASILIA_01-01-2021_A_31-12-2...,DF,A001,BRASILIA,8760,0,2021-01-01,2021-12-31,8759,576,62.8,2023.2
3,2022,INMET_CO_DF_A001_BRASILIA_01-01-2022_A_31-12-2...,DF,A001,BRASILIA,8760,0,2022-01-01,2022-12-31,8759,492,46.2,1356.4
4,2023,INMET_CO_DF_A001_BRASILIA_01-01-2023_A_31-12-2...,DF,A001,BRASILIA,8760,0,2023-01-01,2023-12-31,8755,431,23.4,996.0
5,2024,INMET_CO_DF_A001_BRASILIA_01-01-2024_A_31-12-2...,DF,A001,BRASILIA,8784,0,2024-01-01,2024-12-31,8760,604,35.2,1398.6


In [128]:
teste_parser.to_csv(
    PROCESSED_QUALITY_DIR
    / "teste_parser_inmet_a001_2019_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Teste do parser salvo.")

Teste do parser salvo.


In [129]:
def horas_esperadas_ano(ano):

    if pd.Timestamp(
        year=ano,
        month=12,
        day=31
    ).is_leap_year:

        return 8784

    return 8760

In [130]:
for ano in ANOS_PROJETO:
    print(
        ano,
        horas_esperadas_ano(ano)
    )

2019 8760
2020 8784
2021 8760
2022 8760
2023 8760
2024 8784


In [131]:
def criar_datetime_utc(df):

    resultado = df.copy()

    hora = (
        resultado["hora_utc"]
        .astype("string")
        .str.zfill(4)
    )

    hora_num = pd.to_numeric(
        hora.str[:2],
        errors="coerce"
    )

    minuto_num = pd.to_numeric(
        hora.str[2:],
        errors="coerce"
    )

    resultado["datetime_utc"] = (
        resultado["data"]
        +
        pd.to_timedelta(
            hora_num,
            unit="h"
        )
        +
        pd.to_timedelta(
            minuto_num,
            unit="m"
        )
    )

    return resultado

In [132]:
df_teste_datetime = criar_datetime_utc(
    df_teste
)

print("Datetime inválidos:")
print(
    df_teste_datetime[
        "datetime_utc"
    ].isna().sum()
)

print("\nDatetime duplicados:")
print(
    df_teste_datetime[
        "datetime_utc"
    ].duplicated().sum()
)

print("\nPrimeiro datetime:")
print(
    df_teste_datetime[
        "datetime_utc"
    ].min()
)

print("\nÚltimo datetime:")
print(
    df_teste_datetime[
        "datetime_utc"
    ].max()
)

Datetime inválidos:
0

Datetime duplicados:
0

Primeiro datetime:
2024-01-01 00:00:00

Último datetime:
2024-12-31 23:00:00


In [133]:
UFS_PROJETO = {
    "DF",
    "GO",
    "MS",
    "MT",
    "PR",
    "RS",
    "SC"
}

In [134]:
validacao_parser = []
erros_parser = []

for ano in ANOS_PROJETO:

    print(
        f"\nValidando {ano}..."
    )

    caminho_zip = zips_por_ano[ano]

    with zipfile.ZipFile(
        caminho_zip
    ) as z:

        csvs = [
            nome
            for nome in z.namelist()
            if nome.lower().endswith(".csv")
        ]

        for nome in csvs:

            try:

                info = inspecionar_membro_inmet(
                    z,
                    nome
                )

                meta = info[
                    "metadados"
                ]

                uf = buscar_metadado(
                    meta,
                    "uf"
                ).upper()

                # ignora UFs fora do projeto
                if uf not in UFS_PROJETO:
                    continue

                df_temp, mapa_temp = ler_membro_inmet(
                    z,
                    nome,
                    ano
                )

                df_temp = criar_datetime_utc(
                    df_temp
                )

                horas_esperadas = (
                    horas_esperadas_ano(ano)
                )

                registros = len(
                    df_temp
                )

                validacao_parser.append(
                    {
                        "ano": ano,
                        "uf": uf,
                        "codigo_wmo":
                            df_temp[
                                "codigo_wmo"
                            ].iloc[0],

                        "estacao":
                            df_temp[
                                "estacao"
                            ].iloc[0],

                        "arquivo":
                            nome,

                        "registros":
                            registros,

                        "horas_esperadas":
                            horas_esperadas,

                        "cobertura_temporal_pct":
                            round(
                                registros
                                / horas_esperadas
                                * 100,
                                2
                            ),

                        "datas_invalidas":
                            int(
                                df_temp[
                                    "data"
                                ].isna().sum()
                            ),

                        "datetime_invalidos":
                            int(
                                df_temp[
                                    "datetime_utc"
                                ].isna().sum()
                            ),

                        "datetime_duplicados":
                            int(
                                df_temp[
                                    "datetime_utc"
                                ].duplicated().sum()
                            ),

                        "primeira_data":
                            df_temp[
                                "data"
                            ].min(),

                        "ultima_data":
                            df_temp[
                                "data"
                            ].max(),

                        "precipitacao_validos":
                            int(
                                df_temp[
                                    "precipitacao_mm"
                                ].notna().sum()
                            ),

                        "eventos_chuva":
                            int(
                                (
                                    df_temp[
                                        "precipitacao_mm"
                                    ] > 0
                                ).sum()
                            ),

                        "precipitacao_total_mm":
                            df_temp[
                                "precipitacao_mm"
                            ].sum(
                                min_count=1
                            )
                    }
                )

            except Exception as erro:

                erros_parser.append(
                    {
                        "ano": ano,
                        "arquivo": nome,
                        "erro": str(erro)
                    }
                )

    print(
        f"{ano} concluído."
    )


Validando 2019...
2019 concluído.

Validando 2020...
2020 concluído.

Validando 2021...
2021 concluído.

Validando 2022...
2022 concluído.

Validando 2023...
2023 concluído.

Validando 2024...
2024 concluído.


In [135]:
validacao_parser = pd.DataFrame(
    validacao_parser
)

erros_parser = pd.DataFrame(
    erros_parser
)

print("Arquivos validados:")
print(
    len(validacao_parser)
)

print("\nErros:")
print(
    len(erros_parser)
)

display(
    validacao_parser.head()
)

Arquivos validados:
1195

Erros:
0


,ano,uf,codigo_wmo,estacao,arquivo,registros,horas_esperadas,cobertura_temporal_pct,datas_invalidas,datetime_invalidos,datetime_duplicados,primeira_data,ultima_data,precipitacao_validos,eventos_chuva,precipitacao_total_mm
0,2019,DF,A001,BRASILIA,2019/INMET_CO_DF_A001_BRASILIA_01-01-2019_A_31...,8760,8760,100.0,0,0,0,2019-01-01,2019-12-31,8744,485,1369.4
1,2019,DF,A042,BRAZLANDIA,2019/INMET_CO_DF_A042_BRAZLANDIA_01-01-2019_A_...,8760,8760,100.0,0,0,0,2019-01-01,2019-12-31,8463,419,1466.2
2,2019,DF,A045,AGUAS EMENDADAS,2019/INMET_CO_DF_A045_AGUAS EMENDADAS_01-01-20...,8760,8760,100.0,0,0,0,2019-01-01,2019-12-31,8710,444,1369.6
3,2019,DF,A046,GAMA (PONTE ALTA),2019/INMET_CO_DF_A046_GAMA (PONTE ALTA)_01-01-...,8760,8760,100.0,0,0,0,2019-01-01,2019-12-31,8759,517,1247.8
4,2019,DF,A047,PARANOA (COOPA-DF),2019/INMET_CO_DF_A047_PARANOA (COOPA-DF)_01-01...,8760,8760,100.0,0,0,0,2019-01-01,2019-12-31,8723,454,1187.4


In [136]:
print("Arquivos com datas inválidas:")
print(
    (
        validacao_parser[
            "datas_invalidas"
        ] > 0
    ).sum()
)

print("\nArquivos com datetime inválido:")
print(
    (
        validacao_parser[
            "datetime_invalidos"
        ] > 0
    ).sum()
)

print("\nArquivos com datetime duplicado:")
print(
    (
        validacao_parser[
            "datetime_duplicados"
        ] > 0
    ).sum()
)

print("\nCobertura temporal:")
display(
    validacao_parser[
        "cobertura_temporal_pct"
    ].describe()
)

Arquivos com datas inválidas:
0

Arquivos com datetime inválido:
0

Arquivos com datetime duplicado:
0

Cobertura temporal:


count    1195.000000
mean       99.789540
std         3.468483
min         6.580000
25%       100.000000
50%       100.000000
75%       100.000000
max       100.000000
Name: cobertura_temporal_pct, dtype: float64

In [137]:
cobertura_baixa = (
    validacao_parser[
        validacao_parser[
            "cobertura_temporal_pct"
        ] < 80
    ]
    .sort_values(
        "cobertura_temporal_pct"
    )
)

print(
    "Estação-ano com cobertura < 80%:"
)

print(
    len(cobertura_baixa)
)

display(
    cobertura_baixa.head(30)
)

Estação-ano com cobertura < 80%:
4


,ano,uf,codigo_wmo,estacao,arquivo,registros,horas_esperadas,cobertura_temporal_pct,datas_invalidas,datetime_invalidos,datetime_duplicados,primeira_data,ultima_data,precipitacao_validos,eventos_chuva,precipitacao_total_mm
789,2022,RS,B807,PORTO ALEGRE- BELEM NOVO,INMET_S_RS_B807_PORTO ALEGRE- BELEM NOVO_08-12...,576,8760,6.58,0,0,0,2022-12-08,2022-12-31,556,23,35.2
179,2019,RS,A887,CAPAO DO LEAO (PELOTAS),2019/INMET_S_RS_A887_CAPAO DO LEAO (PELOTAS)_1...,4008,8760,45.75,0,0,0,2019-07-18,2019-12-31,3979,406,762.4
114,2019,MT,A944,ROSARIO OESTE,2019/INMET_CO_MT_A944_ROSARIO OESTE_30-05-2019...,5184,8760,59.18,0,0,0,2019-05-30,2019-12-31,5141,142,453.0
113,2019,MT,A943,SERRA NOVA DOURADA,2019/INMET_CO_MT_A943_SERRA NOVA DOURADA_29-03...,6672,8760,76.16,0,0,0,2019-03-29,2019-12-31,2612,64,204.8


In [138]:
estacao_ano_contagem = (
    validacao_parser
    .groupby(
        [
            "codigo_wmo",
            "ano"
        ]
    )
    .size()
    .sort_values(
        ascending=False
    )
)

print("Combinações únicas codigo_wmo + ano:")
print(
    len(estacao_ano_contagem)
)

print("\nCombinações com mais de um arquivo:")
print(
    (
        estacao_ano_contagem > 1
    ).sum()
)

print("\nMaior quantidade de arquivos para uma mesma estação-ano:")
print(
    estacao_ano_contagem.max()
)

Combinações únicas codigo_wmo + ano:
1195

Combinações com mais de um arquivo:
0

Maior quantidade de arquivos para uma mesma estação-ano:
1


In [139]:
estacoes_ano_multiplos_arquivos = (
    estacao_ano_contagem[
        estacao_ano_contagem > 1
    ]
    .reset_index(
        name="quantidade_arquivos"
    )
)

display(
    estacoes_ano_multiplos_arquivos
)

,codigo_wmo,ano,quantidade_arquivos


In [140]:
detalhes_multiplos = (
    validacao_parser
    .merge(
        estacoes_ano_multiplos_arquivos[
            [
                "codigo_wmo",
                "ano"
            ]
        ],
        on=[
            "codigo_wmo",
            "ano"
        ],
        how="inner"
    )
    .sort_values(
        [
            "codigo_wmo",
            "ano",
            "primeira_data"
        ]
    )
)

display(
    detalhes_multiplos[
        [
            "ano",
            "uf",
            "codigo_wmo",
            "estacao",
            "primeira_data",
            "ultima_data",
            "registros",
            "cobertura_temporal_pct",
            "arquivo"
        ]
    ]
)

,ano,uf,codigo_wmo,estacao,primeira_data,ultima_data,registros,cobertura_temporal_pct,arquivo


In [141]:
def classificar_cobertura(valor):

    if valor >= 95:
        return "excelente"

    elif valor >= 80:
        return "adequada"

    else:
        return "insuficiente"


validacao_parser[
    "qualidade_cobertura"
] = (
    validacao_parser[
        "cobertura_temporal_pct"
    ]
    .apply(
        classificar_cobertura
    )
)

print(
    validacao_parser[
        "qualidade_cobertura"
    ]
    .value_counts()
)

qualidade_cobertura
excelente       1188
insuficiente       4
adequada           3
Name: count, dtype: int64


In [142]:
validacao_parser.to_csv(
    PROCESSED_QUALITY_DIR
    / "validacao_parser_completo_2019_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

erros_parser.to_csv(
    PROCESSED_QUALITY_DIR
    / "erros_parser_completo_2019_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

cobertura_baixa.to_csv(
    PROCESSED_QUALITY_DIR
    / "estacoes_cobertura_temporal_baixa.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Validação completa salva.")

Validação completa salva.


# Processamento definitivo do INMET

Após a validação dos 1.195 arquivos estação-ano, inicia-se a construção
das camadas climáticas processadas.

A camada RAW permanece imutável.

Nesta etapa serão geradas:

- `daily`: indicadores meteorológicos por estação e dia;
- `station_year`: indicadores meteorológicos por estação e ano.

A camada horária continuará representada pelos próprios arquivos RAW,
evitando duplicação desnecessária de milhões de registros.

In [143]:
def soma_min_count(serie):
    return serie.sum(min_count=1)

In [144]:
def gerar_diario_inmet(df):

    dados = criar_datetime_utc(
        df
    )

    dados = dados.copy()

    # Fallback para temperatura máxima/mínima
    dados["_temperatura_max_base"] = (
        dados["temperatura_max_c"]
        .combine_first(
            dados["temperatura_c"]
        )
    )

    dados["_temperatura_min_base"] = (
        dados["temperatura_min_c"]
        .combine_first(
            dados["temperatura_c"]
        )
    )

    daily = (
        dados
        .groupby(
            "data",
            as_index=False
        )
        .agg(
            precipitacao_total_mm=(
                "precipitacao_mm",
                soma_min_count
            ),

            temperatura_media_c=(
                "temperatura_c",
                "mean"
            ),

            temperatura_maxima_c=(
                "_temperatura_max_base",
                "max"
            ),

            temperatura_minima_c=(
                "_temperatura_min_base",
                "min"
            ),

            umidade_media_pct=(
                "umidade_pct",
                "mean"
            ),

            umidade_max_pct=(
                "umidade_max_pct",
                "max"
            ),

            umidade_min_pct=(
                "umidade_min_pct",
                "min"
            ),

            radiacao_total_kj_m2=(
                "radiacao_kj_m2",
                soma_min_count
            ),

            rajada_maxima_ms=(
                "rajada_max_ms",
                "max"
            ),

            velocidade_vento_media_ms=(
                "velocidade_vento_ms",
                "mean"
            ),

            horas_registradas=(
                "datetime_utc",
                "count"
            ),

            horas_precipitacao_validas=(
                "precipitacao_mm",
                "count"
            ),

            horas_temperatura_validas=(
                "temperatura_c",
                "count"
            ),

            horas_umidade_validas=(
                "umidade_pct",
                "count"
            )
        )
    )

    # metadados estação-ano
    daily["codigo_wmo"] = (
        dados["codigo_wmo"].iloc[0]
    )

    daily["estacao"] = (
        dados["estacao"].iloc[0]
    )

    daily["uf"] = (
        dados["uf"].iloc[0]
    )

    daily["regiao"] = (
        dados["regiao"].iloc[0]
    )

    daily["ano"] = (
        dados["ano"].iloc[0]
    )

    daily["latitude"] = (
        dados["latitude"].iloc[0]
    )

    daily["longitude"] = (
        dados["longitude"].iloc[0]
    )

    daily["altitude_m"] = (
        dados["altitude_m"].iloc[0]
    )

    # coberturas diárias
    daily[
        "cobertura_precipitacao_pct"
    ] = (
        daily[
            "horas_precipitacao_validas"
        ]
        / 24
        * 100
    )

    daily[
        "cobertura_temperatura_pct"
    ] = (
        daily[
            "horas_temperatura_validas"
        ]
        / 24
        * 100
    )

    daily[
        "cobertura_umidade_pct"
    ] = (
        daily[
            "horas_umidade_validas"
        ]
        / 24
        * 100
    )

    # chuva só é definida quando existe
    # pelo menos alguma observação válida
    daily["choveu"] = pd.NA

    mascara_valida = (
        daily[
            "horas_precipitacao_validas"
        ] > 0
    )

    daily.loc[
        mascara_valida,
        "choveu"
    ] = (
        daily.loc[
            mascara_valida,
            "precipitacao_total_mm"
        ] > 0
    )

    ordem = [
        "codigo_wmo",
        "estacao",
        "uf",
        "regiao",
        "ano",
        "data",
        "latitude",
        "longitude",
        "altitude_m",
        "precipitacao_total_mm",
        "temperatura_media_c",
        "temperatura_maxima_c",
        "temperatura_minima_c",
        "umidade_media_pct",
        "umidade_max_pct",
        "umidade_min_pct",
        "radiacao_total_kj_m2",
        "rajada_maxima_ms",
        "velocidade_vento_media_ms",
        "choveu",
        "horas_registradas",
        "horas_precipitacao_validas",
        "horas_temperatura_validas",
        "horas_umidade_validas",
        "cobertura_precipitacao_pct",
        "cobertura_temperatura_pct",
        "cobertura_umidade_pct"
    ]

    return daily[ordem]

In [145]:
ZIP_2019 = zips_por_ano[2019]

with zipfile.ZipFile(
    ZIP_2019
) as z:

    csvs_2019 = [
        nome
        for nome in z.namelist()
        if nome.lower().endswith(".csv")
    ]

    nome_a001 = [
        nome
        for nome in csvs_2019
        if "A001" in nome.upper()
    ][0]

    a001_hourly, _ = ler_membro_inmet(
        z,
        nome_a001,
        2019
    )


a001_daily = gerar_diario_inmet(
    a001_hourly
)

print("Dias:")
print(len(a001_daily))

print("\nPeríodo:")
print(
    a001_daily["data"].min(),
    "até",
    a001_daily["data"].max()
)

print("\nDias com chuva:")
print(
    (
        a001_daily["choveu"] == True
    ).sum()
)

print("\nPrecipitação anual:")
print(
    a001_daily[
        "precipitacao_total_mm"
    ].sum(
        min_count=1
    )
)

print("\nMaior precipitação diária:")
print(
    a001_daily[
        "precipitacao_total_mm"
    ].max()
)

display(
    a001_daily.head()
)

Dias:
365

Período:
2019-01-01 00:00:00 até 2019-12-31 00:00:00

Dias com chuva:
131

Precipitação anual:
1369.4

Maior precipitação diária:
72.4


,codigo_wmo,estacao,uf,regiao,ano,data,latitude,longitude,altitude_m,precipitacao_total_mm,...,rajada_maxima_ms,velocidade_vento_media_ms,choveu,horas_registradas,horas_precipitacao_validas,horas_temperatura_validas,horas_umidade_validas,cobertura_precipitacao_pct,cobertura_temperatura_pct,cobertura_umidade_pct
0,A001,BRASILIA,DF,,2019,2019-01-01,-15.789343,-47.925756,1160.96,1.4,...,6.7,1.591667,True,24,24,24,24,100.0,100.0,100.0
1,A001,BRASILIA,DF,,2019,2019-01-02,-15.789343,-47.925756,1160.96,0.0,...,8.3,2.070833,False,24,24,24,24,100.0,100.0,100.0
2,A001,BRASILIA,DF,,2019,2019-01-03,-15.789343,-47.925756,1160.96,0.0,...,7.0,2.108333,False,24,24,24,24,100.0,100.0,100.0
3,A001,BRASILIA,DF,,2019,2019-01-04,-15.789343,-47.925756,1160.96,0.0,...,7.9,1.833333,False,24,24,24,24,100.0,100.0,100.0
4,A001,BRASILIA,DF,,2019,2019-01-05,-15.789343,-47.925756,1160.96,1.0,...,12.0,1.25,True,24,24,24,24,100.0,100.0,100.0


In [146]:
def gerar_station_year(
    hourly,
    daily
):

    dados = criar_datetime_utc(
        hourly
    )

    ano = int(
        dados["ano"].iloc[0]
    )

    horas_esperadas = (
        horas_esperadas_ano(
            ano
        )
    )

    dados["_temperatura_max_base"] = (
        dados["temperatura_max_c"]
        .combine_first(
            dados["temperatura_c"]
        )
    )

    dados["_temperatura_min_base"] = (
        dados["temperatura_min_c"]
        .combine_first(
            dados["temperatura_c"]
        )
    )

    horas_registradas = len(
        dados
    )

    precipitacao_validos = (
        dados[
            "precipitacao_mm"
        ]
        .notna()
        .sum()
    )

    temperatura_validos = (
        dados[
            "temperatura_c"
        ]
        .notna()
        .sum()
    )

    umidade_validos = (
        dados[
            "umidade_pct"
        ]
        .notna()
        .sum()
    )

    cobertura_temporal_pct = (
        horas_registradas
        / horas_esperadas
        * 100
    )

    cobertura_precipitacao_pct = (
        precipitacao_validos
        / horas_esperadas
        * 100
    )

    cobertura_temperatura_pct = (
        temperatura_validos
        / horas_esperadas
        * 100
    )

    cobertura_umidade_pct = (
        umidade_validos
        / horas_esperadas
        * 100
    )

    resultado = {
        "codigo_wmo":
            dados[
                "codigo_wmo"
            ].iloc[0],

        "estacao":
            dados[
                "estacao"
            ].iloc[0],

        "uf":
            dados[
                "uf"
            ].iloc[0],

        "regiao":
            dados[
                "regiao"
            ].iloc[0],

        "ano":
            ano,

        "latitude":
            dados[
                "latitude"
            ].iloc[0],

        "longitude":
            dados[
                "longitude"
            ].iloc[0],

        "altitude_m":
            dados[
                "altitude_m"
            ].iloc[0],

        "precipitacao_anual_mm":
            dados[
                "precipitacao_mm"
            ].sum(
                min_count=1
            ),

        "temperatura_media_anual_c":
            dados[
                "temperatura_c"
            ].mean(),

        "temperatura_maxima_anual_c":
            dados[
                "_temperatura_max_base"
            ].max(),

        "temperatura_minima_anual_c":
            dados[
                "_temperatura_min_base"
            ].min(),

        "umidade_media_anual_pct":
            dados[
                "umidade_pct"
            ].mean(),

        "radiacao_anual_kj_m2":
            dados[
                "radiacao_kj_m2"
            ].sum(
                min_count=1
            ),

        "rajada_maxima_anual_ms":
            dados[
                "rajada_max_ms"
            ].max(),

        "velocidade_vento_media_anual_ms":
            dados[
                "velocidade_vento_ms"
            ].mean(),

        "dias_com_chuva":
            int(
                (
                    daily[
                        "precipitacao_total_mm"
                    ] > 0
                ).sum()
            ),

        "dias_precipitacao_valida":
            int(
                daily[
                    "precipitacao_total_mm"
                ]
                .notna()
                .sum()
            ),

        "dias_observados":
            int(
                len(daily)
            ),

        "horas_registradas":
            int(
                horas_registradas
            ),

        "horas_esperadas":
            int(
                horas_esperadas
            ),

        "cobertura_temporal_pct":
            round(
                cobertura_temporal_pct,
                2
            ),

        "cobertura_precipitacao_pct":
            round(
                cobertura_precipitacao_pct,
                2
            ),

        "cobertura_temperatura_pct":
            round(
                cobertura_temperatura_pct,
                2
            ),

        "cobertura_umidade_pct":
            round(
                cobertura_umidade_pct,
                2
            ),

        "qualidade_cobertura":
            classificar_cobertura(
                cobertura_temporal_pct
            )
    }

    return pd.DataFrame(
        [resultado]
    )

In [147]:
a001_station_year = (
    gerar_station_year(
        a001_hourly,
        a001_daily
    )
)

display(
    a001_station_year
)

,codigo_wmo,estacao,uf,regiao,ano,latitude,longitude,altitude_m,precipitacao_anual_mm,temperatura_media_anual_c,...,dias_com_chuva,dias_precipitacao_valida,dias_observados,horas_registradas,horas_esperadas,cobertura_temporal_pct,cobertura_precipitacao_pct,cobertura_temperatura_pct,cobertura_umidade_pct,qualidade_cobertura
0,A001,BRASILIA,DF,,2019,-15.789343,-47.925756,1160.96,1369.4,21.964078,...,131,365,365,8760,8760,100.0,99.82,99.82,99.82,excelente


In [154]:
relatorio_processamento = []

print(
    "Relatório de processamento reiniciado."
)

Relatório de processamento reiniciado.


In [155]:
relatorio_processamento = []

for ano in ANOS_PROJETO:

    print(
        f"\n{'=' * 60}"
    )

    print(
        f"PROCESSANDO {ano}"
    )

    print(
        f"{'=' * 60}"
    )

    caminho_zip = (
        zips_por_ano[
            ano
        ]
    )

    diarios_ano = []
    station_year_ano = []

    with zipfile.ZipFile(
        caminho_zip
    ) as z:

        csvs = [
            nome
            for nome in z.namelist()
            if nome.lower().endswith(
                ".csv"
            )
        ]

        for indice, nome in enumerate(
            csvs,
            start=1
        ):

            try:

                info = inspecionar_membro_inmet(
                    z,
                    nome
                )

                meta = info[
                    "metadados"
                ]

                uf = buscar_metadado(
                    meta,
                    "uf"
                ).upper()

                if uf not in UFS_PROJETO:
                    continue

                hourly, _ = ler_membro_inmet(
                    z,
                    nome,
                    ano
                )

                daily = gerar_diario_inmet(
                    hourly
                )

                station_year = (
                    gerar_station_year(
                        hourly,
                        daily
                    )
                )

                diarios_ano.append(
                    daily
                )

                station_year_ano.append(
                    station_year
                )

                relatorio_processamento.append(
                    {
                        "ano": ano,
                        "uf": uf,
                        "codigo_wmo":
                            station_year[
                                "codigo_wmo"
                            ].iloc[0],

                        "estacao":
                            station_year[
                                "estacao"
                            ].iloc[0],

                        "arquivo":
                            nome,

                        "status":
                            "sucesso",

                        "registros_horarios":
                            len(hourly),

                        "registros_diarios":
                            len(daily),

                        "cobertura_temporal_pct":
                            station_year[
                                "cobertura_temporal_pct"
                            ].iloc[0],

                        "cobertura_precipitacao_pct":
                            station_year[
                                "cobertura_precipitacao_pct"
                            ].iloc[0]
                    }
                )

            except Exception as erro:

                relatorio_processamento.append(
                    {
                        "ano": ano,
                        "uf": None,
                        "codigo_wmo": None,
                        "estacao": None,
                        "arquivo": nome,
                        "status": "erro",
                        "erro": str(erro)
                    }
                )

    daily_ano_df = pd.concat(
        diarios_ano,
        ignore_index=True
    )

    station_year_ano_df = pd.concat(
        station_year_ano,
        ignore_index=True
    )

    caminho_daily = (
        PROCESSED_DAILY_DIR
        / f"inmet_daily_{ano}.csv"
    )

    caminho_station_year = (
        PROCESSED_STATION_YEAR_DIR
        / f"inmet_station_year_{ano}.csv"
    )

    daily_ano_df.to_csv(
        caminho_daily,
        index=False,
        encoding="utf-8-sig"
    )

    station_year_ano_df.to_csv(
        caminho_station_year,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "Daily:",
        daily_ano_df.shape
    )

    print(
        "Station-year:",
        station_year_ano_df.shape
    )

    print(
        "Arquivos salvos."
    )

    del diarios_ano
    del station_year_ano
    del daily_ano_df
    del station_year_ano_df


PROCESSANDO 2019


C:\Users\Elaine Cristina\AppData\Local\Temp\ipykernel_28520\2642484756.py:139: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  daily_ano_df = pd.concat(
C:\Users\Elaine Cristina\AppData\Local\Temp\ipykernel_28520\2642484756.py:144: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  station_year_ano_df = pd.concat(


Daily: (75708, 27)
Station-year: (209, 26)
Arquivos salvos.

PROCESSANDO 2020


C:\Users\Elaine Cristina\AppData\Local\Temp\ipykernel_28520\2642484756.py:139: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  daily_ano_df = pd.concat(
C:\Users\Elaine Cristina\AppData\Local\Temp\ipykernel_28520\2642484756.py:144: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  station_year_ano_df = pd.concat(


Daily: (76128, 27)
Station-year: (208, 26)
Arquivos salvos.

PROCESSANDO 2021


...

Daily: (69009, 27)
Station-year: (190, 26)
Arquivos salvos.

PROCESSANDO 2023


C:\Users\Elaine Cristina\AppData\Local\Temp\ipykernel_28520\2642484756.py:139: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  daily_ano_df = pd.concat(
C:\Users\Elaine Cristina\AppData\Local\Temp\ipykernel_28520\2642484756.py:144: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  station_year_ano_df = pd.concat(


Daily: (69715, 27)
Station-year: (191, 26)
Arquivos salvos.

PROCESSANDO 2024


C:\Users\Elaine Cristina\AppData\Local\Temp\ipykernel_28520\2642484756.py:139: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  daily_ano_df = pd.concat(
C:\Users\Elaine Cristina\AppData\Local\Temp\ipykernel_28520\2642484756.py:144: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  station_year_ano_df = pd.concat(


Daily: (69540, 27)
Station-year: (190, 26)
Arquivos salvos.


In [150]:
ZIP_2019 = zips_por_ano[2019]

with zipfile.ZipFile(
    ZIP_2019
) as z:

    csvs_2019 = [
        nome
        for nome in z.namelist()
        if nome.lower().endswith(".csv")
    ]

    nome_a001 = [
        nome
        for nome in csvs_2019
        if "A001" in nome.upper()
    ][0]

    teste_regiao, _ = ler_membro_inmet(
        z,
        nome_a001,
        2019
    )


display(
    teste_regiao[
        [
            "codigo_wmo",
            "estacao",
            "uf",
            "regiao",
            "latitude",
            "longitude"
        ]
    ].head()
)

,codigo_wmo,estacao,uf,regiao,latitude,longitude
0,A001,BRASILIA,DF,centro_oeste,-15.789343,-47.925756
1,A001,BRASILIA,DF,centro_oeste,-15.789343,-47.925756
2,A001,BRASILIA,DF,centro_oeste,-15.789343,-47.925756
3,A001,BRASILIA,DF,centro_oeste,-15.789343,-47.925756
4,A001,BRASILIA,DF,centro_oeste,-15.789343,-47.925756


In [151]:
print(
    teste_regiao[
        [
            "uf",
            "regiao"
        ]
    ]
    .drop_duplicates()
)

   uf        regiao
0  DF  centro_oeste


In [152]:
with zipfile.ZipFile(
    ZIP_2019
) as z:

    csvs_2019 = [
        nome
        for nome in z.namelist()
        if nome.lower().endswith(".csv")
    ]

    nome_sul = [
        nome
        for nome in csvs_2019
        if "_SC_" in nome.upper()
    ][0]

    teste_sul, _ = ler_membro_inmet(
        z,
        nome_sul,
        2019
    )


display(
    teste_sul[
        [
            "codigo_wmo",
            "estacao",
            "uf",
            "regiao",
            "latitude",
            "longitude"
        ]
    ].head()
)

,codigo_wmo,estacao,uf,regiao,latitude,longitude
0,A806,FLORIANOPOLIS,SC,sul,-27.60253,-48.620096
1,A806,FLORIANOPOLIS,SC,sul,-27.60253,-48.620096
2,A806,FLORIANOPOLIS,SC,sul,-27.60253,-48.620096
3,A806,FLORIANOPOLIS,SC,sul,-27.60253,-48.620096
4,A806,FLORIANOPOLIS,SC,sul,-27.60253,-48.620096


In [153]:
print(
    teste_sul[
        [
            "uf",
            "regiao"
        ]
    ]
    .drop_duplicates()
)

   uf regiao
0  SC    sul


In [156]:
arquivos_station_year = sorted(
    PROCESSED_STATION_YEAR_DIR.glob(
        "inmet_station_year_*.csv"
    )
)

print(
    "Arquivos station_year encontrados:",
    len(arquivos_station_year)
)

for arquivo in arquivos_station_year:

    print(
        arquivo.name
    )

Arquivos station_year encontrados: 6
inmet_station_year_2019.csv
inmet_station_year_2020.csv
inmet_station_year_2021.csv
inmet_station_year_2022.csv
inmet_station_year_2023.csv
inmet_station_year_2024.csv


In [157]:
station_year_final = pd.concat(
    [
        pd.read_csv(
            arquivo,
            dtype={
                "codigo_wmo": str,
                "uf": str,
                "regiao": str
            }
        )
        for arquivo in arquivos_station_year
    ],
    ignore_index=True
)

print(
    "Dimensão final:"
)

print(
    station_year_final.shape
)

Dimensão final:
(1195, 26)


In [158]:
print(
    "Regiões ausentes:"
)

print(
    station_year_final[
        "regiao"
    ]
    .isna()
    .sum()
)


print(
    "\nRegiões vazias:"
)

print(
    (
        station_year_final[
            "regiao"
        ]
        .fillna("")
        .str.strip()
        == ""
    )
    .sum()
)

Regiões ausentes:
0

Regiões vazias:
0


In [159]:
display(
    station_year_final[
        "regiao"
    ]
    .value_counts(
        dropna=False
    )
)

regiao
centro_oeste    631
sul             564
Name: count, dtype: int64

In [160]:
display(
    station_year_final[
        [
            "uf",
            "regiao"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "uf"
    )
    .reset_index(
        drop=True
    )
)

,uf,regiao
0,DF,centro_oeste
1,GO,centro_oeste
2,MS,centro_oeste
3,MT,centro_oeste
4,PR,sul
5,RS,sul
6,SC,sul


In [161]:
duplicidades_station_year = (
    station_year_final[
        [
            "codigo_wmo",
            "ano"
        ]
    ]
    .duplicated()
    .sum()
)

print(
    "Duplicidades codigo_wmo + ano:"
)

print(
    duplicidades_station_year
)

Duplicidades codigo_wmo + ano:
0


In [162]:
display(
    station_year_final[
        "ano"
    ]
    .value_counts()
    .sort_index()
)

ano
2019    209
2020    208
2021    207
2022    190
2023    191
2024    190
Name: count, dtype: int64

In [163]:
print(
    len(
        station_year_final
    )
)

1195


In [164]:
display(
    station_year_final[
        [
            "ano",
            "horas_esperadas"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "ano"
    )
    .reset_index(
        drop=True
    )
)

,ano,horas_esperadas
0,2019,8760
1,2020,8784
2,2021,8760
3,2022,8760
4,2023,8760
5,2024,8784


In [165]:
relatorio_processamento_df = pd.DataFrame(
    relatorio_processamento
)

print(
    "Total de arquivos processados:"
)

print(
    len(
        relatorio_processamento_df
    )
)


print(
    "\nStatus:"
)

display(
    relatorio_processamento_df[
        "status"
    ]
    .value_counts(
        dropna=False
    )
)

Total de arquivos processados:
1195

Status:


status
sucesso    1195
Name: count, dtype: int64

In [166]:
relatorio_processamento_df.to_csv(
    PROCESSED_QUALITY_DIR
    / "relatorio_processamento_inmet_2019_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Relatório de processamento salvo."
)

Relatório de processamento salvo.


# ✅ Encerramento — Processamento climático INMET

A etapa de processamento dos dados meteorológicos do INMET foi concluída para o período de **2019 a 2024**.

## Fonte utilizada

Foram utilizados exclusivamente os arquivos ZIP originais disponibilizados pelo INMET, mantidos de forma imutável na camada RAW:

`data/raw/inmet/{ano}/arquivo_inmet_{ano}.zip`

Os arquivos foram lidos diretamente dos ZIPs, sem necessidade de extração permanente.

## Recorte territorial

Nesta etapa foram consideradas as UFs pertencentes ao escopo climático atual do projeto:

- DF
- GO
- MT
- MS
- PR
- RS
- SC

Correspondentes às regiões:

- Centro-Oeste
- Sul

## Período

**2019–2024**

## Validações realizadas

Foram verificadas:

- integridade dos arquivos ZIP;
- estrutura dos arquivos CSV do INMET;
- identificação automática do cabeçalho;
- metadados das estações;
- código WMO;
- UF;
- estação;
- latitude;
- longitude;
- altitude;
- datas e horários;
- registros duplicados;
- cobertura temporal;
- disponibilidade de precipitação;
- coerência das regiões geográficas.

O parser foi validado em todos os arquivos pertencentes ao escopo do projeto.

### Resultado da validação

- Estações-ano processadas: **1.195**
- Erros de parsing: **0**
- Combinações duplicadas `codigo_wmo + ano`: **0**
- Regiões identificadas corretamente:
  - Centro-Oeste: **631 estação-ano**
  - Sul: **564 estação-ano**

## Bases geradas

Foram produzidas bases intermediárias nas camadas `databases_processed/inmet`.

### Daily

Granularidade:

`estação × dia`

Contém indicadores climáticos diários derivados das observações horárias.

### Station Year

Granularidade:

`estação × ano`

Contém indicadores climáticos anuais, incluindo:

- precipitação anual;
- temperatura média;
- temperatura máxima;
- temperatura mínima;
- umidade;
- radiação;
- vento;
- dias com chuva;
- cobertura dos dados.

## Próxima etapa

A base `station_year` ainda representa estações meteorológicas.

Para permitir a integração com:

- IBGE/PAM;
- MapBiomas;
- INPE Queimadas;
- MapBiomas Solo;
- SEEG;

será necessário transformar a unidade espacial:

`estação meteorológica → município IBGE`

Essa etapa será executada no notebook:

**05_integracao_inmet_malha_municipal.ipynb**

A associação será realizada utilizando as coordenadas de cada estação em cada ano e a Malha Municipal do IBGE.

> O processamento climático INMET é considerado concluído neste notebook.